# TRAITEMENT DES DONNES FUSIONNEES DE LA PREMIERE FEUILLE EXCEL

**Objectif** : Traitement/apurement des variables démographiques
* Date de naissance 
* Sexe
* Situation Matrimoniale
* Nombre d'enfants

Nous aurons également pour mission de créer de nouvelles variables à partir des informations disponibles. Il s'agit du statut dans l'emploi (Fonctionnaire ou contractuelle), l'âge, l'age de première prise de service, l'ancienneté dans l'organisme, nombre d'année d'expérience.

Un contrôle sera effectué sur le rattachement administratif:

* Matricule
* Fonction
* Services
* Organismes
* Situation dans l'emploi
* Emplois
* Fonction
* Classes/Echelons
* Lieu d'affectation

## Chargement des packages necessaires

In [2]:
pip install fastparquet

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Paramètres
import io
import pandas as pd
import boto3
import io
import unicodedata
import re
import numpy as np
import unicodedata
import unicodedata, re
import re

# FONCTIONS NECESSAIRES

### Situation matrimoniale

In [4]:
import pandas as pd
import numpy as np

def build_situation_matrimoniale(
    df: pd.DataFrame,
    source_col: str = "SITUATION MATRIMONIALE",
    out_col: str = "SITUATION_MATRIMONIALE_RECODE",
    mapping: dict | None = None,
    return_report: bool = True
):
    """
    Recoder la situation matrimoniale + tableau effectifs/pourcentages.
    - mapping: dictionnaire de recodage (par défaut celui que tu as donné).
    - out_col: nom de la colonne recodée.
    Retourne: (df, report) si return_report=True, sinon df.
    """
    if mapping is None:
        mapping = {
            "Célibataire": "Célibataire",
            "Cas Particulier": "Autres",
            "Divorcé Séparé": "Divorcé/Séparé",
            "Femme Mariée": "Marié(e)",
            "Invalide Marié": "Marié(e)",
            "Marié": "Marié(e)",
            "Veuf / veuve": "Veuf/Veuve"
        }

    out = df.copy()
    if source_col not in out.columns:
        raise KeyError(f"Colonne '{source_col}' absente du DataFrame.")

    # Recodage
    out[out_col] = out[source_col].replace(mapping)

    # Tableaux de synthèse
    eff = out[out_col].value_counts(dropna=False).sort_index()
    pct = out[out_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    report = {
        "table": pd.DataFrame(
            {"Effectif": eff, "Pourcentage (%)": pct.round(2)}
        ),
        "value_counts": eff,
        "value_counts_pct": pct.round(2)
    }

    return (out, report) if return_report else out

### Nombre d'enfants

In [5]:
def build_nombre_enfants(
    df: pd.DataFrame,
    source_col: str = "NOMBRE ENFANT",
    out_col: str = "NBRE_ENFTS_IMPUTE",
    fill_value: int | float = 0,
    return_report: bool = True
):
    """
    Créer la variable NBRE_ENFTS_IMPUTE:
    - copie de la variable d'origine,
    - conversion numérique (coercition), NaN -> fill_value (0 par défaut),
    - tableaux effectifs/pourcentages.
    Retourne: (df, report) si return_report=True, sinon df.
    """
    out = df.copy()
    if source_col not in out.columns:
        raise KeyError(f"Colonne '{source_col}' absente du DataFrame.")

    # Conversion -> numérique puis imputation des manquants à 0
    out[out_col] = pd.to_numeric(out[source_col], errors="coerce")
    out[out_col] = out[out_col].fillna(fill_value)

    # Si ce sont bien des nombres entiers, cast en Int64 (garde les NA si jamais)
    try:
        out[out_col] = out[out_col].round(0).astype("Int64")
    except Exception:
        # Si des décimaux existent vraiment, on reste en Float64
        out[out_col] = out[out_col].astype("Float64")

    # Tableaux de synthèse
    eff = out[out_col].value_counts(dropna=False).sort_index()
    pct = out[out_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    report = {
        "table": pd.DataFrame(
            {"Effectif": eff, "Pourcentage (%)": pct.round(2)}
        ),
        "value_counts": eff,
        "value_counts_pct": pct.round(2)
    }

    return (out, report) if return_report else out

### CREATION DE LA VARIABLE CATEGORIE 1 ET 2 

On crée deux variables catégorie à partir des deux variables grades

**CATEGORIE_1**  
- Extraite de la colonne `GRADE`.  
- Si le libellé contient « non fonctionnaire », l’individu est classé **Non Fonctionnaire**.  
- Sinon, la lettre **A, B, C ou D** est récupérée depuis des formes comme « Catégorie A7 » ou « Catégorie C ».  
- Les valeurs non reconnues sont remplacées par **Non Fonctionnaire**.

**CATEGORIE_2**  
- Dérivée de la colonne `GRADE 2`.  
- Si le code est alphanumérique de type **A7, B3, C1, D2**, on conserve la lettre initiale (**A–D**).  
- Si c’est déjà une lettre seule (**A–D**), elle est gardée telle quelle.  
- Toute autre valeur est classée **Non Fonctionnaire**.

In [6]:
def build_categories(
    df: pd.DataFrame,
    grade_col="GRADE",
    grade2_col="GRADE 2",
    compute_categorie2=True,     # calcule CATEGORIE_2 depuis GRADE 2 si la colonne existe
    overwrite_categorie2=True    # réécrit CATEGORIE_2 même si déjà présente
):
    """
    - CATEGORIE_1 depuis GRADE (texte):
        'Non Fonctionnaire' si le texte contient 'non fonctionnaire' (insensible à la casse/espaces)
        sinon extrait la lettre A-D depuis 'Catégorie A7', 'Catégorie B', etc.
        NaN -> 'Non Fonctionnaire'
    - CATEGORIE_2 depuis GRADE 2 (lettres A-D):
        'A7' -> 'A', 'B' -> 'B', autres -> NaN -> 'Non Fonctionnaire'
    - Les 2 colonnes sont typées 'category' avec les mêmes modalités ordonnées.

    Retourne: df (copie) avec CATEGORIE_1 (et CATEGORIE_2 si demandé).
    """

    out = df.copy()
    cat_order = ["Non Fonctionnaire", "A", "B", "C", "D"]

    # ---------- helpers ----------
    def _cat_from_grade2_letter(val):
        """ Cette fonction transforme un GRADE 2 en catégorie """
        if pd.isna(val):
            return pd.NA  # si valeur manquante, on retourne pd.NA.
        s = str(val).strip().upper()  # on convertit en chaîne, on enlève les espaces et on met en majuscules
        if re.fullmatch(r"[ABCD]\d+", s):   # A7, B3, C1, D2...
            return s[0]
        if re.fullmatch(r"[ABCD]", s):      # A, B, C, D
            return s
        return pd.NA

    def _parse_cat_from_grade_text(val):
        if pd.isna(val):
            return pd.NA
        t = str(val)
        # ex.: "Non   fonctionnaire", "non-fonctionnaire"
        if re.search(r"\bnon\s*fonctionnaire\b", t, flags=re.I):
            return "Non Fonctionnaire"
        # ex.: "Catégorie A", "Categorie B7", "catégorie c titulaire"
        m = re.search(r"Cat[ée]gorie\s+([A-D])(?:\s*\d+)?", t, flags=re.I)
        if m:
            return m.group(1).upper()
        return pd.NA

    # ---------- CATEGORIE_1 depuis GRADE ----------
    if grade_col not in out.columns:
        raise KeyError(f"Colonne '{grade_col}' absente.")
    out["CATEGORIE_1"] = out[grade_col].apply(_parse_cat_from_grade_text)
    out["CATEGORIE_1"] = out["CATEGORIE_1"].fillna("Non Fonctionnaire") # Remplace les NaN par "Non Fonctionnaire"
    out["CATEGORIE_1"] = pd.Categorical(out["CATEGORIE_1"], categories=cat_order, ordered=True) #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    # ---------- (optionnel) CATEGORIE_2 depuis GRADE 2 ----------
    if compute_categorie2 and (grade2_col in out.columns):
        if overwrite_categorie2 or ("CATEGORIE_2" not in out.columns):
            cat2_raw = out[grade2_col].apply(_cat_from_grade2_letter)
            cat2 = cat2_raw.astype(object)
            cat2[pd.isna(cat2)] = "Non Fonctionnaire" # Remplace les NaN par "Non Fonctionnaire"
            out["CATEGORIE_2"] = pd.Categorical(cat2, categories=cat_order, ordered=True) # #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    return out 

### CREATION DE LA VARIABLE STATUT

On crée une variable `STATUT` à partir des catégories `CATEGORIE_1` et `CATEGORIE_2` (détectées automatiquement parmi les colonnes candidates) et de `EMPLOI_NORM`.

Les libellés sont normalisés pour identifier **Non Fonctionnaire** ou une lettre **A–D**.

**La règle de décision est :**
- **Cas particulier** : si l’emploi est renseigné **et** qu’au moins une des deux catégories est *Non Fonctionnaire* ; **ou** si l’emploi est vide mais qu’au moins une des deux catégories est une lettre **A–D**.
- **Fonctionnaire** : si l’emploi est renseigné **et** que les **deux** catégories sont des lettres **A–D**.
- **Non Fonctionnaire** : si l’emploi est vide **et** que les **deux** catégories sont *Non Fonctionnaire*.
- Sinon : **Non Fonctionnaire** (par défaut).


In [7]:
def build_statut_from_cats(df: pd.DataFrame, 
                           emploi_col: str = "EMPLOI_NORM",
                           cat1_col: str | None = None,
                           cat1_candidates: tuple[str,str] = ("CATEGORIE_1", "CATEGORIE 1"),
                           cat2_col: str | None = None,
                           cat2_candidates: tuple[str,str] = ("CATEGORIE_2", "CATEGORIE 2"),
                           label_cas: str = "Cas particulier"):

    import unicodedata, re
    import numpy as np
    import pandas as pd

    out = df.copy()

    # --- Trouver les colonnes ---
    def _pick_col(cand_list):
        for c in cand_list:
            if c in out.columns:
                return c
        return None

    if cat1_col is None:
        cat1_col = _pick_col(cat1_candidates)
    if cat2_col is None:
        cat2_col = _pick_col(cat2_candidates)

    for col, name in zip([cat1_col, cat2_col, emploi_col], ["CATEGORIE_1","CATEGORIE_2","EMPLOI_NORM"]):
        if col is None or col not in out.columns:
            raise KeyError(f"Colonne '{name}' introuvable.")

    # --- EMPLOI renseigné ---
    emp_norm = out[emploi_col].fillna("").astype(str).str.strip()
    has_emploi = emp_norm.ne("")

    # --- Normalisation catégories ---
    def _norm_ascii_lower(x):
        if pd.isna(x):
            return ""
        s = str(x).strip()
        s = unicodedata.normalize("NFKD", s)
        s = s.encode("ascii", "ignore").decode("ascii")
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    c1_norm = out[cat1_col].map(_norm_ascii_lower)
    c2_norm = out[cat2_col].map(_norm_ascii_lower)

    c1_is_nf = c1_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)
    c2_is_nf = c2_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)

    c1_is_abcd = c1_norm.str.fullmatch(r"[abcd]", na=False)
    c2_is_abcd = c2_norm.str.fullmatch(r"[abcd]", na=False)

    is_nf_any = c1_is_nf | c2_is_nf
    is_nf_all = c1_is_nf & c2_is_nf
    is_abcd_any = c1_is_abcd | c2_is_abcd
    is_abcd_all = c1_is_abcd & c2_is_abcd

    # --- STATUT ---
    statut = np.select(
        [(has_emploi & is_nf_any) | (~has_emploi & is_abcd_any),
         has_emploi & is_abcd_all,
         ~has_emploi & is_nf_all],
        [label_cas, "Fonctionnaire", "Non Fonctionnaire"],
        default="Non Fonctionnaire"
    ).astype(object)

    out["STATUT"] = pd.Categorical(
        statut,
        categories=["Non Fonctionnaire", "Fonctionnaire", label_cas],
        ordered=True
    )

    # --- Rapport automatique ---
    rep = {
        "repartition_statut": out["STATUT"].value_counts().reindex(
            ["Non Fonctionnaire","Fonctionnaire",label_cas], fill_value=0
        )
    }

    return out, rep

### CREATION DE LA VARIABLE STATUT DEFINITIF 

Créer une variable Statut_def qui renseigne le statut définitif de l'individu sur une période donnée. En effet, un individu peut être fonctionnaire sur un organisme et ne pas contractuel sur un autre organisme. Le matricule permet d'identifier de manière unique un individu. On combinant le matricule et le code organisme on peut retrouver le statut de l'individu sur chaque organisme où il intervient. Pour une période de collecte donnée (mois et année), le statut définitif sera qu'il est fonctionnaire si sur au moins un organisme l'individu est fonctionnaire sinon non fonctionnaire. Attention, le contrôle se fait sur la période. Ici nous avons 72 périodes (janvier 2015 à décembre 2020).

In [8]:
def build_statut_def_periode(
    df: pd.DataFrame,
    statut_col: str = "STATUT",
    matricule_col: str = "MATRICULE",
    periode_col: str = "PERIODE",
    output_col: str = "Statut_def_mois",
    return_report: bool = True
):

    """
        Règle:
      - Pour chaque groupe (MATRICULE et PERIODE ), on définit le statut final:
          * Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
          * Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
          * Sinon → 'Non Fonctionnaire'
          
          Colonnes attendues: statut_col, matricule_col, periode_col
    """
    out = df.copy()

    # Vérifications
    for col in [statut_col, matricule_col, periode_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Normalisation
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Agrégation par groupe. On regroupe par MATRICULE et PERIODE pour vérifier si au moins
    # une ligne est “Fonctionnaire” ou “Cas particulier”.
    key_cols = [matricule_col, periode_col]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Priorité pour le statut final
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        grp_summary = out.groupby(key_cols)[output_col].first()
        rep = {
            "Statut définifitif": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0),
        }
        return out, rep

    return out

In [9]:
def build_statut_def_annee(df, 
                            statut_col="STATUT", 
                            matricule_col="MATRICULE",
                            date_collecte_col="DATE_COLLECTE",
                            output_col="Statut_def_annee",
                            return_report=True):
    """
    Détermine le statut définitif par MATRICULE et ANNEE (extraite de DATE_COLLECTE).
    
    Règle:
      - Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
      - Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
      - Sinon → 'Non Fonctionnaire'
    
    Retour:
        - df mis à jour
        - report (distribution par Statut_def) si return_report=True
    """
    out = df.copy()

    # Vérifications colonnes
    for col in [statut_col, matricule_col, date_collecte_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Extraire l'année
    out[date_collecte_col] = pd.to_datetime(out[date_collecte_col], errors="coerce")
    out["ANNEE"] = out[date_collecte_col].dt.year

    # Normalisation du statut
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Regroupement par matricule et année
    key_cols = [matricule_col, "ANNEE"]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Détermination du statut définitif
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        rep = {
            "Statut_def_distribution": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0)
        }
        return out, rep

    return out


### CREATION DE LA VARIABLE SEXE CLEAN ET IMPUTE

On crée deux variables normalisées du sexe à savoir :

**SEXE_CLEAN**  
- Extraite de `SEXE` en prenant, pour chaque `MATRICULE`, la **dernière valeur non nulle** selon `DATE_COLLECTE`.  
- En cas d’égalité parfaite sur la date, on retient la **dernière ligne** présente.  
- Si aucune valeur valide n’existe pour l’individu, la valeur est **NaN**.

**SEXE_IMPUTE**  
- Dérivée de `SEXE_CLEAN`.  
- Si `SEXE_CLEAN` est manquant, imputation par le **mode** observé au même **mois × année** de collecte (`MOIS_COLLECTE`, `ANNEE_COLLECTE`).  
- Si `SEXE_CLEAN` est renseigné, la valeur est **conservée** telle quelle.  
- Si aucun mode n’est disponible pour la période, la valeur **reste manquante**.



In [10]:
import pandas as pd
import numpy as np

def build_sexe_clean(
    df: pd.DataFrame,
    id_col="MATRICULE",
    sex_col="SEXE",
    collect_col="DATE_COLLECTE",
    new_col="SEXE_CLEAN"
):
    """
    Crée une variable propre du sexe (SEXE_CLEAN) en prenant, pour chaque individu (id_col),
    la dernière valeur non nulle observée selon la date collect_col.

    Règles:
      - Utilise uniquement les lignes avec date ET sexe non nuls pour déterminer la "dernière" valeur.
      - En cas d’égalité parfaite sur la date, on prend la dernière occurrence dans l’ordre des lignes.
      - Si un individu n’a jamais de sexe non nul (ou pas de date valable), SEXE_CLEAN = NaN.
    """
    df = df.copy()
    # Sécuriser la date
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # Ordre de ligne pour casser les ex æquo au besoin (stable, reproductible)
    df["_row_order"] = np.arange(len(df))

    # Garder uniquement les lignes exploitables pour définir la "dernière" info
    base = (
        df.dropna(subset=[collect_col, sex_col])
          .sort_values([id_col, collect_col, "_row_order"])
    )

    # Prendre la dernière valeur (par date puis par ordre de ligne)
    latest = (
        base.groupby(id_col, as_index=False)
            .last()[[id_col, sex_col]]
            .rename(columns={sex_col: new_col})
    )

    # Fusionner dans le df complet (toutes les lignes récupèrent la même valeur propre par id)
    df = df.drop(columns=[new_col], errors="ignore").merge(latest, on=id_col, how="left")

    # Nettoyage
    df = df.drop(columns=["_row_order"])

    return df


def imputer_sexe(
    df: pd.DataFrame,
    sex_col="SEXE_CLEAN",         # ← on impute désormais à partir de SEXE_CLEAN
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_IMPUTE",
    debug=True
):
    """
    Impute les valeurs manquantes de `sex_col` en utilisant le mode par (ANNEE_COLLECTE × MOIS_COLLECTE),
    et stocke le résultat dans `sex_valid_col`.
    """
    import pandas as pd
    import numpy as np
    from IPython.display import display

    df = df.copy()
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # Clés temporelles
    df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    df["MOIS_COLLECTE"]  = df[collect_col].dt.month

    # Colonne imputée initialisée avec le propre (SEXE_CLEAN)
    df[sex_valid_col] = df[sex_col]

    # Mode par groupe (année × mois) calculé sur la variable propre
    mode_sexe_groupes = (
        df.dropna(subset=[sex_col])
          .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])[sex_col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    # Imputation: si NaN, remplacer par le mode du groupe; sinon, garder la valeur
    df[sex_valid_col] = df.apply(
        lambda row: mode_sexe_groupes.get(
            (row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), row[sex_valid_col]
        ) if pd.isna(row[sex_valid_col]) else row[sex_valid_col],
        axis=1
    )

    # Stats
    tab_sexe_avant = df[sex_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres = df[sex_valid_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres_pct = df[sex_valid_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    crosstab_sexe = pd.crosstab(
        df[sex_col], df[sex_valid_col],
        dropna=False, margins=True, margins_name="Total"
    )

    report = {
        "tab_sexe_avant": tab_sexe_avant,
        "tab_sexe_apres": tab_sexe_apres,
        "tab_sexe_apres_pct": tab_sexe_apres_pct,
        "crosstab_sexe": crosstab_sexe
    }

    if debug:
        print("=== Statistiques avant imputation (SEXE_CLEAN) ===")
        display(tab_sexe_avant.to_frame("Effectif"))
        print("\n=== Statistiques après imputation (SEXE_IMPUTE) ===")
        display(tab_sexe_apres.to_frame("Effectif"))
        print("\n=== Pourcentages après imputation ===")
        display(tab_sexe_apres_pct.to_frame("Pourcentage (%)"))
        print("\n=== Tableau croisé avant/après ===")
        display(crosstab_sexe)

    return df, report


### CREATION DE LA VARIABLE AGE IMPUTE

On crée des variables d’âge à partir des dates de naissance et de collecte.

**DATE_NAISSANCE_CLEAN**  
- Nettoyage de `DATE NAISSANCE` (formats mixtes, zéros, numéros Excel) → conversion en date valide.

**ANNEE_NAISSANCE / MOIS_NAISSANCE / JOUR_NAISSANCE**  
- Composantes extraites de `DATE_NAISSANCE_CLEAN`.

**DATE_NAISSANCE_CORRIGEE**  
- Pour chaque `MATRICULE`, on retient **la date de naissance la plus récente** observée (selon `DATE_COLLECTE`).  
- Composantes associées : `ANNEE_NAISSANCE_CORRIGEE`, `MOIS_NAISSANCE_CORRIGEE`, `JOUR_NAISSANCE_CORRIGEE`.

**AGE**  
- Âge calculé à la date de collecte (`DATE_COLLECTE`) à partir de `DATE_NAISSANCE_CORRIGEE` (années pleines).

**AGE_VALIDE**  
- Filtrage de `AGE` dans l’intervalle **[18 ; 70]** inclus ; sinon valeur manquante.

**AGE_IMPUTE**  
- Si `AGE_VALIDE` est manquant : imputation par la **médiane** du groupe **année × mois** (et **sexe** si `SEXE_IMPUTE` existe).  
- Si la médiane de groupe est indisponible : repli sur la **médiane globale**.  
- Résultat arrondi à l’entier (type `Int64`).

In [11]:
def _to_datetime_mixed(s):
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

def _clean_dates_generic(series):
    s = series.copy()
    s_str = s.astype(str).str.strip().str.lower()
    time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
    zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
    empties   = s_str.isin(["", "nan", "nat", "none", "nul", "null"])
    s = s.mask(time_only | zero_date | empties, np.nan)
    dt = _to_datetime_mixed(s)
    serial = pd.to_numeric(s_str, errors="coerce")
    serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
    if serial_mask.any():
        dt_serial = pd.to_datetime(serial[serial_mask], unit="D",
                                   origin="1899-12-30", errors="coerce")
        dt.loc[serial_mask] = dt_serial
    return dt

def _mode_safe(s: pd.Series):
    s_nonan = s.dropna()
    if s_nonan.empty: 
        return pd.NA
    m = s_nonan.mode()
    return m.iloc[0] if not m.empty else pd.NA


# ============================================================
# 1) ÂGE OBSERVÉ / IMPUTÉ / STABILISÉ
# ============================================================
def build_age_valide(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    age_min=18, age_max=70,
    do_impute_age=True,
    stabilize_age=True,
    stabilize_method="last"   # 'last' | 'mode' | 'median'
):
    """
    Construit AGE, AGE_VALIDE, AGE_IMPUTE et AGE_IMPUTE_STABLE (optionnel).
    - Nettoie dates.
    - Pour la naissance, retient la + récente par matricule.
    - AGE précis (anniversaire).
    - Impute AGE par médiane (ANNEE×MOIS×SEXE si dispo).
    - Stabilise AGE_IMPUTE par matricule (dernière/mode/médiane).
    """
    df = df.copy()
    df.columns = pd.Index(df.columns).map(str).str.strip()

    # --- Collecte
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # --- Naissance
    df["DATE_NAISSANCE_CLEAN"] = _clean_dates_generic(df[birth_col])
    # la + récente observée
    tmp = (
        df.dropna(subset=["DATE_NAISSANCE_CLEAN"])
          .sort_values([matricule_col, collect_col], ascending=[True, False])
          .drop_duplicates(subset=[matricule_col], keep="first")
          [[matricule_col, "DATE_NAISSANCE_CLEAN"]]
          .rename(columns={"DATE_NAISSANCE_CLEAN": "DATE_NAISSANCE_CORRIGEE"})
    )
    df = df.drop(columns=["DATE_NAISSANCE_CORRIGEE"], errors="ignore")
    df = df.merge(tmp, on=matricule_col, how="left", validate="many_to_one")
    df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(df["DATE_NAISSANCE_CORRIGEE"], errors="coerce")

    # --- AGE précis à la collecte
    birth = df["DATE_NAISSANCE_CORRIGEE"]
    ref   = df[collect_col]
    age = pd.Series(pd.NA, index=df.index, dtype="Float64")
    mask = birth.notna() & ref.notna()
    if mask.any():
        ydiff = ref[mask].dt.year - birth[mask].dt.year
        before_bday = (ref[mask].dt.month < birth[mask].dt.month) | (
            (ref[mask].dt.month == birth[mask].dt.month) & (ref[mask].dt.day < birth[mask].dt.day)
        )
        age.loc[mask] = (ydiff - before_bday.astype(int)).astype("Float64")
    df["AGE"] = age
    df["AGE_VALIDE"] = df["AGE"].where(df["AGE"].between(age_min, age_max))

    # --- AGE_IMPUTE (médiane ANNEE×MOIS×SEXE si dispo)
    if do_impute_age:
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE"]
        if sex_col in df.columns:
            group_keys.append(sex_col)
        med = df.groupby(group_keys)["AGE_VALIDE"].transform("median")
        global_med = df["AGE_VALIDE"].median()
        df["AGE_IMPUTE"] = (
            df["AGE_VALIDE"].where(df["AGE_VALIDE"].notna(), med.fillna(global_med))
            .round(0).astype("Int64")
        )

    # --- Stabilisation par matricule
    if stabilize_age:
        g = df.sort_values([matricule_col, collect_col]).groupby(matricule_col)["AGE_IMPUTE"]
        if stabilize_method == "last":
            stable = g.last()
        elif stabilize_method == "mode":
            stable = g.apply(_mode_safe)
        elif stabilize_method == "median":
            stable = g.median()
        else:
            raise ValueError("stabilize_method ∈ {'last','mode','median'}")
        df = df.merge(stable.rename("AGE_IMPUTE_STABLE"), on=matricule_col, how="left")

        # Bilan rapide
        nb_multi_age = df.groupby(matricule_col)["AGE_IMPUTE"].nunique(dropna=True).gt(1).sum()
        tot = df[matricule_col].nunique()
        print(f"[AGE] Matricules avec AGE_IMPUTE variable : {nb_multi_age:,} ({nb_multi_age/tot*100:.2f}%)")
        print(f"[AGE] Stabilisation : {stabilize_method}")

    return df
    

### CREATION DE LA VARIABLE AGE DE PRISE DE SERVICE 


On crée des variables d’**âge au service** à partir de la prise de service et de la date de collecte.

**ANNEE_COLLECTE / MOIS_COLLECTE**  
- Extraits de `DATE_COLLECTE` (créés si absents) pour structurer les calculs et l’imputation.

**PRISE_DE_SERVICE_CLEAN**  
- Nettoyage de `PRISE DE SERVICE` (formats mixtes, zéros, numéros Excel) → date valide.

**PRISE_DE_SERVICE_CORRIGEE**  
- Pour chaque `MATRICULE`, on retient la valeur de `PRISE_DE_SERVICE_CLEAN` observée sur la **collecte la plus récente** (si `recompute_corrected=True`), sinon on conserve la colonne existante.

**AGE_AU_SERVICE**  
- Calculé pour les lignes où `AGE_IMPUTE` et `PRISE_DE_SERVICE_CORRIGEE` sont présents :  
  `AGE_AU_SERVICE = AGE_IMPUTE - ⌊(DATE_COLLECTE - PRISE_DE_SERVICE_CORRIGEE) / 365⌋` (approx. en années).

**AGE_AU_SERVICE_VALIDE**  
- Filtrage de `AGE_AU_SERVICE` dans l’intervalle **[18 ; 40]** inclus ; sinon valeur manquante.

**AGE_AU_SERVICE_IMPUTE**  
- Si `AGE_AU_SERVICE_VALIDE` est manquant : imputation par la **médiane** du groupe **ANNEE_COLLECTE × MOIS_COLLECTE × SEXE_IMPUTE**,  
  avec repli sur la **médiane globale** ; résultat arrondi (type Float64).

**Rapport (optionnel)**  
- Tables avant/après et échantillon d’anomalies de prise de service si `return_tables=True`.


In [12]:
def build_age_service(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    age_min=18, age_max=40,
    days_per_year=365,
    age_src_priority=("AGE_IMPUTE_STABLE","AGE_IMPUTE","AGE"),  # ← ordre de préférence
    stabilize_service=True,
    stabilize_method="mode",  # 'last' | 'mode' | 'median'
    return_tables=True,
    sample_anomalies=10,
    recompute_corrected=True
):
    """
    Construit l'âge au service et le stabilise.

    Sorties principales:
      - PRISE_DE_SERVICE_CLEAN, PRISE_DE_SERVICE_CORRIGEE
      - AGE_AU_SERVICE, AGE_AU_SERVICE_VALIDE, AGE_AU_SERVICE_IMPUTE
      - FLAG_MULTI_AGE_SERVICE
      - AGE_AU_SERVICE_FINAL (si stabilize_service=True)
    Logique:
      1) Nettoyage PDS + correction = valeur observée la plus récente par matricule (si recompute_corrected=True)
      2) Choix de la source d'âge selon age_src_priority (première colonne existante)
      3) AGE_AU_SERVICE = âge_source - floor((collecte - PDS_corrigée)/365)
      4) Bornage [age_min; age_max] → AGE_AU_SERVICE_VALIDE
      5) Imputation des manquants par médiane (ANNEE×MOIS×SEXE si dispo) sinon médiane globale
      6) Stabilisation par matricule (mode/last/median) → AGE_AU_SERVICE_FINAL
    """
    df = df.copy()
    df.columns = pd.Index(df.columns).map(str).str.strip()

    # --- 0) DATE_COLLECTE & dérivées
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if year_collect_col not in df.columns:
        df[year_collect_col] = df[collect_col].dt.year
    if month_collect_col not in df.columns:
        df[month_collect_col] = df[collect_col].dt.month

    # --- 1) PRISE_DE_SERVICE_CLEAN
    df["PRISE_DE_SERVICE_CLEAN"] = _clean_dates_generic(df[start_col_raw])

    # --- 2) Diagnostics PDS multiples
    nb_dates_par_matricule = df.groupby(matricule_col)["PRISE_DE_SERVICE_CLEAN"].nunique(dropna=True)
    matricules_problemes = nb_dates_par_matricule[nb_dates_par_matricule > 1].index.tolist()
    anomalies_dates = (
        df[df[matricule_col].isin(matricules_problemes)]
        [[matricule_col, collect_col, "PRISE_DE_SERVICE_CLEAN"]]
        .sort_values([matricule_col, collect_col])
    )

    # --- 3) PRISE_DE_SERVICE_CORRIGEE = valeur la + récente observée
    if recompute_corrected or (start_col_corrected not in df.columns):
        panel_sorted = df.sort_values([matricule_col, collect_col], ascending=[True, False])
        ps_corr = (
            panel_sorted.drop_duplicates(subset=[matricule_col], keep="first")
            [[matricule_col, "PRISE_DE_SERVICE_CLEAN"]]
            .rename(columns={"PRISE_DE_SERVICE_CLEAN": start_col_corrected})
        )
        df = df.drop(columns=[start_col_corrected], errors="ignore")
        df = df.merge(ps_corr, on=matricule_col, how="left", validate="many_to_one")

    df[start_col_corrected] = _to_datetime_mixed(df[start_col_corrected])

    # --- 4) Colonnes de sortie (âge au service)
    df["AGE_AU_SERVICE"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
    df["AGE_AU_SERVICE_VALIDE"] = pd.Series(pd.NA, index=df.index, dtype="Float64")

    # --- 5) Choix de la source d'âge (priorité)
    age_src = None
    for col in age_src_priority:
        if col in df.columns:
            age_src = col
            break
    if age_src is None:
        age_src = "_AGE_SRC_TMP_"
        df[age_src] = pd.Series(pd.NA, index=df.index, dtype="Float64")

    df[age_src] = pd.to_numeric(df[age_src], errors="coerce").astype("Float64")

    # --- 6) Calcul AGE_AU_SERVICE quand âge & PDS sont présents
    mask = df[age_src].notna() & df[start_col_corrected].notna()
    if mask.any():
        delta_days = (df.loc[mask, collect_col] - df.loc[mask, start_col_corrected]).dt.days
        df.loc[mask, "AGE_AU_SERVICE"] = (
            df.loc[mask, age_src] - (delta_days // days_per_year)
        ).astype("Float64")

    # --- 7) Bornage de l'âge au service
    ok = df["AGE_AU_SERVICE"].between(age_min, age_max)
    df.loc[ok, "AGE_AU_SERVICE_VALIDE"] = df.loc[ok, "AGE_AU_SERVICE"].astype("Float64")

    # --- 8) Imputation (ANNEE×MOIS×SEXE si dispo ; sinon médiane globale)
    candidate_keys = [year_collect_col, month_collect_col, sex_col]
    group_keys = [k for k in candidate_keys if (k is not None and k in df.columns)]
    global_med = df["AGE_AU_SERVICE_VALIDE"].median()
    if len(group_keys) > 0:
        med_group = df.groupby(group_keys)["AGE_AU_SERVICE_VALIDE"].transform("median")
    else:
        med_group = pd.Series(global_med, index=df.index, dtype="Float64")

    impute_base = df["AGE_AU_SERVICE_VALIDE"].where(
        df["AGE_AU_SERVICE_VALIDE"].notna(), med_group.fillna(global_med)
    )
    df["AGE_AU_SERVICE_IMPUTE"] = pd.to_numeric(impute_base, errors="coerce").round(0).astype("Float64")

    # --- 9) Drapeau multi-âges imputés (avant stabilisation)
    flag_multi = df.groupby(matricule_col)["AGE_AU_SERVICE_IMPUTE"].nunique(dropna=True) > 1
    df["FLAG_MULTI_AGE_SERVICE"] = df[matricule_col].map(flag_multi).astype("Int64")

    # --- 10) Stabilisation finale par matricule
    if stabilize_service:
        g = (
            df.sort_values([matricule_col, collect_col])
              .groupby(matricule_col)["AGE_AU_SERVICE_IMPUTE"]
        )
        try:
            if stabilize_method == "last":
                final = g.last()
            elif stabilize_method == "mode":
                final = g.apply(_mode_safe)
            elif stabilize_method == "median":
                final = g.median()
            else:
                raise ValueError("stabilize_method ∈ {'last','mode','median'}")
            df = df.merge(final.rename("AGE_AU_SERVICE_FINAL"), on=matricule_col, how="left")
        except Exception:
            # Filet de sécurité : retomber sur l'imputé
            df["AGE_AU_SERVICE_FINAL"] = df["AGE_AU_SERVICE_IMPUTE"]
    else:
        df["AGE_AU_SERVICE_FINAL"] = df["AGE_AU_SERVICE_IMPUTE"]

    # >>> Garde supplémentaire : garantir l'existence de la colonne <<<
    if "AGE_AU_SERVICE_FINAL" not in df.columns:
        df["AGE_AU_SERVICE_FINAL"] = df["AGE_AU_SERVICE_IMPUTE"]

    # --- 11) Rapport (optionnel)
    if return_tables:
        tot = df[matricule_col].nunique()
        before = df.groupby(matricule_col)["AGE_AU_SERVICE_IMPUTE"].nunique(dropna=True).gt(1).sum()
        after  = df.groupby(matricule_col)["AGE_AU_SERVICE_FINAL"].nunique(dropna=True).gt(1).sum()
        pct_b = round(before / tot * 100, 2)
        pct_a = round(after / tot * 100, 2)

        print(f"[SERVICE] Multi-âges imputés AVANT stabilisation : {before:,} / {tot:,} ({pct_b}%)")
        print(f"[SERVICE] Multi-âges APRÈS  stabilisation : {after:,} / {tot:,} ({pct_a}%)  → méthode={stabilize_method}")

        tab_valide = df["AGE_AU_SERVICE_VALIDE"].value_counts(dropna=False).sort_index()
        tab_valide_pct = df["AGE_AU_SERVICE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()
        tab_impute = df["AGE_AU_SERVICE_IMPUTE"].value_counts(dropna=False).sort_index()
        tab_impute_pct = df["AGE_AU_SERVICE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index()

        report = {
            "n_matricules_multi_service_dates": len(matricules_problemes),
            "anomalies_pds_head": anomalies_dates.head(sample_anomalies),
            "tab_age_service_valide": pd.DataFrame({
                "Effectif": tab_valide,
                "Pourcentage (%)": (tab_valide_pct * 100).round(2)
            }),
            "tab_age_service_impute": pd.DataFrame({
                "Effectif": tab_impute,
                "Pourcentage (%)": (tab_impute_pct * 100).round(2)
            }),
            "multi_before_after": {"before": before, "after": after, "total": tot},
            "age_source_used": age_src
        }
        return df, report

    return df


### CREATION DE LA VARIBALE ANCIENNTE DANS L'ORGANISME ET DEPUIS LA PRISE DE FONCTION

#### CREATION DE LA VARIBALE ANCIENNTE  DEPUIS LA PRISE DE FONCTION

On calcule l’ancienneté **mensuelle** sur une fenêtre donnée (ex. 2015-01 → 2020-12), par **personne** et, en option, par **organisme**.

**PERIODE**  
- Mois de référence (début de mois) dérivé de `DATE_COLLECTE` ; sert de clé mensuelle.

**PRISE_DE_SERVICE_FIXE**  
- Date de prise de service retenue **une fois par `MATRICULE`** (la plus ancienne connue).  
- Indicateur associé : `START_AVANT_FENETRE` (=1 si la prise de service est antérieure à `min_period`).

**BASE_PRE2015_MOIS_PERS**  
- Base d’ancienneté **avant** la fenêtre : nombre de mois entre la prise de service et le mois précédent `min_period` (0 si prise de service ≥ fenêtre).  
- Ajoutée seulement si l’option *pré-fenêtre* est activée.

**Ancienneté par PERSONNE (tous organismes confondus)**  
- `NB_APPA_MOIS_PERS` : nombre de lignes observées dans le mois.  
- `PRESENCE_MOIS_PERS` : 1 si présence au moins une fois dans le mois, sinon 0.  
- `ANCIENNETE_MOIS_PERS` : cumul mensuel de `PRESENCE_MOIS_PERS` (dans la fenêtre).  
- `ANCIENNETE_AN_PERS` : `ANCIENNETE_MOIS_PERS // 12`.  
- `ANCIENNETE_MOIS_PERS_TOTAL` : `BASE_PRE2015_MOIS_PERS` + `ANCIENNETE_MOIS_PERS`.  
- `ANCIENNETE_AN_PERS_TOTAL` : `ANCIENNETE_MOIS_PERS_TOTAL // 12`.

**Ancienneté par ORGANISME (si `CODE_ORGANISME` fourni)**  
- `NB_APPA_MOIS_ORG` : nombre de lignes dans le mois pour le couple `MATRICULE × ORGANISME`.  
- `PRESENCE_MOIS_ORG` : 1 si présence au moins une fois dans le mois, sinon 0.  
- `ANCIENNETE_MOIS_ORG` : cumul mensuel de `PRESENCE_MOIS_ORG`.  
- `ANCIENNETE_AN_ORG` : `ANCIENNETE_MOIS_ORG // 12`.

**Notes**  
- La fenêtre est **incluse** : de `min_period` à `max_period`.  
- Si `keep_counts=False`, les colonnes `NB_APPA_MOIS_*` ne sont pas conservées.  
- Le rapport retourne : bornes de période, nombre de matricules et un échantillon illustratif par personne.


In [13]:
def build_anciennete_mensuelle(
    df: pd.DataFrame,
    matricule_col: str = "MATRICULE",
    org_col: str | None = "CODE_ORGANISME",   # mettre None pour ignorer l’organisme
    collect_col: str = "DATE_COLLECTE",
    start_service_col: str = "PRISE_DE_SERVICE_CORRIGEE",  # ne change pas d’une année à l’autre

    # Fenêtre d’observation (ex. 2015–2020)
    min_period: str = "2015-01",   # inclus
    max_period: str = "2020-12",   # inclus

    # Options de sortie
    return_by_org: bool = True,    # ancienneté par organisme si org_col != None
    return_by_person: bool = True, # ancienneté globale par personne (tous organismes)
    keep_counts: bool = True,      # garder NB_APPA_MOIS_* et PRESENCE_MOIS_*

    # Pré-compte avant la fenêtre (si prise de service antérieure à min_period)
    include_prewindow_base: bool = True
):
    """
    Calcule l'ancienneté en MOIS :
      - avant la fenêtre : si include_prewindow_base=True et prise de service < min_period,
        on ajoute une base d'ancienneté continue du mois de prise de service jusqu'au mois précédent min_period.
      - dans la fenêtre [min_period, max_period] : on additionne UNIQUEMENT les mois effectivement observés
        (présence = au moins une ligne dans le mois).

    Sorties dans le DataFrame retourné (jointes au niveau mensuel) :
      - Par PERSONNE : NB_APPA_MOIS_PERS, PRESENCE_MOIS_PERS, ANCIENNETE_MOIS_PERS,
        BASE_PRE2015_MOIS_PERS (si include_prewindow_base),
        ANCIENNETE_MOIS_PERS_TOTAL (= base pré-fenêtre + observé),
        ANCIENNETE_AN_PERS, ANCIENNETE_AN_PERS_TOTAL.
      - Par ORGANISME : NB_APPA_MOIS_ORG, PRESENCE_MOIS_ORG, ANCIENNETE_MOIS_ORG, ANCIENNETE_AN_ORG.
    """
    out = df.copy()

    # ---------- 0) Dates et borne de fenêtre ----------
    out[collect_col] = pd.to_datetime(out[collect_col], errors="coerce")
    if out[collect_col].isna().all():
        raise ValueError(f"Toutes les valeurs de '{collect_col}' sont manquantes ou invalides.")

    # PERIODE = début de mois (clé mensuelle)
    out["PERIODE"] = out[collect_col].dt.to_period("M").dt.to_timestamp(how="start")

    # Fenêtre [min_period, max_period] (début de mois)
    min_p = pd.Period(min_period, freq="M").to_timestamp(how="start")
    max_p = pd.Period(max_period, freq="M").to_timestamp(how="start")

    # Filtre fenêtre
    out = out.loc[(out["PERIODE"] >= min_p) & (out["PERIODE"] <= max_p)].copy()

    # Calendrier mensuel complet pour la fenêtre
    full_months = pd.date_range(min_p, max_p, freq="MS")  # MonthStart (DatetimeIndex)

    # ---------- 1) Prise de service fixe (une par matricule) ----------
    if start_service_col in out.columns:
        tmp_ps = (
            out[[matricule_col, start_service_col]]
            .dropna()
            .sort_values([matricule_col, start_service_col])  # la plus ANCIENNE
            .drop_duplicates(subset=[matricule_col], keep="first")
            .rename(columns={start_service_col: "PRISE_DE_SERVICE_FIXE"})
        )
        out = out.drop(columns=["PRISE_DE_SERVICE_FIXE"], errors="ignore")
        out = out.merge(tmp_ps, on=matricule_col, how="left")
    else:
        out["PRISE_DE_SERVICE_FIXE"] = pd.NaT

    out["PRISE_DE_SERVICE_FIXE"] = pd.to_datetime(out["PRISE_DE_SERVICE_FIXE"], errors="coerce")
    out["START_AVANT_FENETRE"] = out["PRISE_DE_SERVICE_FIXE"].lt(min_p)

    # ---------- 2) Base pré-fenêtre par PERSONNE ----------
    if include_prewindow_base:
        from pandas.tseries.offsets import MonthBegin

        start_window = min_p                      # 1er jour de la fenêtre (ex : 2015-01-01)
        end_pre = start_window - MonthBegin(1)    # 1er jour du mois précédent (ex : 2014-12-01)

        def months_diff_inclusive(d0: pd.Timestamp, d1: pd.Timestamp) -> int:
            """Nb de mois entre d0 et d1, inclusivement, en ignorant les jours."""
            if pd.isna(d0) or pd.isna(d1) or d0 > d1:
                return 0
            return (d1.year - d0.year) * 12 + (d1.month - d0.month) + 1

        base_pre = (
            out[[matricule_col, "PRISE_DE_SERVICE_FIXE"]]
            .dropna(subset=["PRISE_DE_SERVICE_FIXE"])
            .drop_duplicates(subset=[matricule_col], keep="first")
        ).copy()

        base_pre["BASE_PRE2015_MOIS_PERS"] = base_pre["PRISE_DE_SERVICE_FIXE"].apply(
            lambda d: months_diff_inclusive(d, end_pre) if d < start_window else 0
        )

        out = out.merge(
            base_pre[[matricule_col, "BASE_PRE2015_MOIS_PERS"]],
            on=matricule_col, how="left"
        )
        out["BASE_PRE2015_MOIS_PERS"] = out["BASE_PRE2015_MOIS_PERS"].fillna(0).astype(int)
    else:
        out["BASE_PRE2015_MOIS_PERS"] = 0

    # ----------------------------------------------------------------
    # OUTIL : calendrier complet via produit cartésien (évite la perte des colonnes de clé)
    # ----------------------------------------------------------------
    def _anciennete_mensuelle(group_keys_prefix: list[str], suffix: str):
        """
        Construit un tableau mensuel complet par (group_keys_prefix) x PERIODE :
          - NB_APPA_MOIS{suffix} = nombre de lignes dans le mois
          - PRESENCE_MOIS{suffix} = 1 si NB_APPA_MOIS > 0, sinon 0
          - ANCIENNETE_MOIS{suffix} = somme cumulée des PRESENCE_MOIS par groupe
          - ANCIENNETE_AN{suffix} = ANCIENNETE_MOIS{suffix} // 12
        """
        keys = group_keys_prefix + ["PERIODE"]

        # a) Compter les apparitions (toutes lignes) par mois
        counts = (
            out.groupby(keys, dropna=False)
               .size()
               .rename(f"NB_APPA_MOIS{suffix}")
               .reset_index()
        )

        # b) Produit cartésien (groupes × full_months)
        groups = counts[group_keys_prefix].drop_duplicates()
        months_df = pd.DataFrame({"PERIODE": full_months})
        groups["_key"] = 1
        months_df["_key"] = 1
        grid = groups.merge(months_df, on="_key").drop(columns="_key")

        # c) Joindre les comptes et compléter à 0
        filled = grid.merge(counts, on=keys, how="left")
        nb_col = f"NB_APPA_MOIS{suffix}"
        filled[nb_col] = filled[nb_col].fillna(0).astype(int)

        # d) Présence binaire
        pres_col = f"PRESENCE_MOIS{suffix}"
        filled[pres_col] = (filled[nb_col] > 0).astype(int)

        # e) Ancienneté cumulée en mois (dans la fenêtre)
        anc_col = f"ANCIENNETE_MOIS{suffix}"
        filled[anc_col] = (
            filled.sort_values(group_keys_prefix + ["PERIODE"])
                  .groupby(group_keys_prefix, dropna=False)[pres_col]
                  .cumsum()
                  .astype("Int64")
        )

        # f) Années (entières)
        anc_year_col = f"ANCIENNETE_AN{suffix}"
        filled[anc_year_col] = (filled[anc_col] // 12).astype("Int64")

        return filled

    # ---------- 3) Ancienneté par ORGANISME ----------
    df_org = None
    if return_by_org and (org_col is not None) and (org_col in out.columns):
        df_org = _anciennete_mensuelle([matricule_col, org_col], "_ORG")

    # ---------- 4) Ancienneté par PERSONNE ----------
    df_pers = None
    if return_by_person:
        # Construire les comptes à partir de out (tous organismes confondus)
        # → On part de out mais on regroupe seulement par matricule et PERIODE.
        tmp = out[[matricule_col, "PERIODE"]].copy()
        tmp["_ones"] = 1
        counts_pers = (
            tmp.groupby([matricule_col, "PERIODE"], dropna=False)["_ones"]
               .sum()
               .rename("NB_APPA_MOIS_PERS")
               .reset_index()
        )

        # Produit cartésien (matricules × full_months)
        pers = counts_pers[[matricule_col]].drop_duplicates()
        months_df = pd.DataFrame({"PERIODE": full_months})
        pers["_key"] = 1
        months_df["_key"] = 1
        grid_pers = pers.merge(months_df, on="_key").drop(columns="_key")

        df_pers = grid_pers.merge(counts_pers, on=[matricule_col, "PERIODE"], how="left")
        df_pers["NB_APPA_MOIS_PERS"] = df_pers["NB_APPA_MOIS_PERS"].fillna(0).astype(int)

        df_pers["PRESENCE_MOIS_PERS"] = (df_pers["NB_APPA_MOIS_PERS"] > 0).astype(int)
        df_pers["ANCIENNETE_MOIS_PERS"] = (
            df_pers.sort_values([matricule_col, "PERIODE"])
                   .groupby([matricule_col], dropna=False)["PRESENCE_MOIS_PERS"]
                   .cumsum()
                   .astype("Int64")
        )
        df_pers["ANCIENNETE_AN_PERS"] = (df_pers["ANCIENNETE_MOIS_PERS"] // 12).astype("Int64")

    # ---------- 5) Joins sur le niveau mensuel ----------
    base_keys = [matricule_col, "PERIODE"]

    if (return_by_org and (df_org is not None)):
        out = out.merge(df_org, on=[matricule_col, org_col, "PERIODE"], how="left")

    if (return_by_person and (df_pers is not None)):
        out = out.merge(df_pers, on=base_keys, how="left")

        # Ajouter la base pré-fenêtre à l'ancienneté cumulée PERSONNE
        out["ANCIENNETE_MOIS_PERS_TOTAL"] = (
            out["BASE_PRE2015_MOIS_PERS"].astype(int) + out["ANCIENNETE_MOIS_PERS"].fillna(0).astype(int)
        ).astype("Int64")
        out["ANCIENNETE_AN_PERS_TOTAL"] = (out["ANCIENNETE_MOIS_PERS_TOTAL"] // 12).astype("Int64")

    # ---------- 6) Nettoyage colonnes optionnel ----------
    if not keep_counts:
        drop_cols = [c for c in out.columns if c.startswith("NB_APPA_MOIS")]
        out.drop(columns=drop_cols, inplace=True, errors="ignore")

    # ---------- 7) Report synthétique ----------
    rep = {
        "periode_min": min_p.date(),
        "periode_max": max_p.date(),
        "nb_matricules": out[matricule_col].nunique(),
        "exemple_pers": None
    }
    if return_by_person and ("ANCIENNETE_MOIS_PERS" in out.columns):
        rep["exemple_pers"] = (
            out[[matricule_col, "PERIODE", "NB_APPA_MOIS_PERS", "PRESENCE_MOIS_PERS",
                 "ANCIENNETE_MOIS_PERS", "BASE_PRE2015_MOIS_PERS", "ANCIENNETE_MOIS_PERS_TOTAL"]]
            .sort_values([matricule_col, "PERIODE"])
            .head(20)
        )

    return out, rep


#### CREATION DE LA VARIBALE ANCIENNTE DANS L'ORGANISME

On calcule l’ancienneté par **organisme** à partir de la prise de service retenue au **premier mois observé**.

**PERIODE**  
- Mois civil (début de mois) dérivé de `DATE_COLLECTE` ; sert de clé mensuelle.

**DATE_ENTREE_ORG** (`entry_col`)  
- Pour chaque couple `MATRICULE × ID_ORGANISME`, on retient :  
  - La **PRISE DE SERVICE** située sur la **plus vieille** `DATE_COLLECTE` du couple.  
  - Si cette PS est manquante :  
    - `fallback_when_ps_missing="min_ps_non_null"` → on prend la **plus vieille PS non nulle** du couple ;  
    - sinon (par défaut) → on prend la **plus vieille `DATE_COLLECTE`** comme date d’entrée.

**ANCIENNETE_ORG_MOIS** (`months_col`)  
- Écart calendaire en **mois** entre `DATE_ENTREE_ORG` et `PERIODE`.  
- Paramètre `inclusive=True` : le **mois d’entrée compte** (ex. mai→mai = 1) ; si `False`, il ne compte pas.

**ANCIENNETE_ORG_AN** (`years_col`)  
- Conversion en **années entières** : `ANCIENNETE_ORG_MOIS // 12`.

In [14]:
import pandas as pd
import numpy as np

def build_anciennete_organisme_depuis_ps_min_collecte(
    df: pd.DataFrame,
    matricule_col: str = "MATRICULE",
    org_col: str = "ID_ORGANISME",
    collect_col: str = "DATE_COLLECTE",
    ps_col: str = "PRISE DE SERVICE",       # la colonne fournie dans ta base
    entry_col: str = "DATE_ENTREE_ORG",      # date d'entrée retenue (PS à min DATE_COLLECTE)
    months_col: str = "ANCIENNETE_ORG_MOIS",
    years_col: str = "ANCIENNETE_ORG_AN",
    inclusive: bool = True,                  # True => le mois d’entrée compte (mai→mai = 1)
    fallback_when_ps_missing: str = "min_collecte",  
    # "min_collecte" => si la PS est manquante sur la ligne la + vieille, on prend min(DATE_COLLECTE)
    # "min_ps_non_null" => sinon chercher la + vieille PRISE DE SERVICE non nulle du couple
    return_report: bool = True
):
    """
    Logique :
      1) Trouver, pour chaque couple (MATRICULE, ORGANISME), la ligne ayant la plus vieille DATE_COLLECTE.
      2) Prendre la PRISE DE SERVICE de CETTE ligne comme date d'entrée (DATE_ENTREE_ORG).
         - Si cette PS est manquante :
             * fallback_when_ps_missing == "min_ps_non_null" : on prend la plus vieille PS non nulle du couple.
             * sinon (par défaut) : on prend la plus vieille DATE_COLLECTE comme entrée.
      3) Pour chaque ligne, calculer ANCIENNETE_ORG_MOIS = écart calendaire en mois entre DATE_ENTREE_ORG et le mois de la ligne
         (mois d’entrée compté si inclusive=True).
      4) ANCIENNETE_ORG_AN = ANCIENNETE_ORG_MOIS // 12

    Remarque :
      - On n’invente aucune ligne : calcul au niveau des lignes existantes.
    """
    out = df.copy()

    # -- sécuriser dates --
    out[collect_col] = pd.to_datetime(out[collect_col], errors="coerce")
    out[ps_col] = pd.to_datetime(out[ps_col], errors="coerce")
    if out[collect_col].isna().all():
        raise ValueError(f"Toutes les valeurs de '{collect_col}' sont manquantes ou invalides.")

    # PERIODE = mois civil (début de mois)
    out["PERIODE"] = out[collect_col].dt.to_period("M").dt.to_timestamp(how="start")

    # 1) identifier, par couple, la ligne à min DATE_COLLECTE
    base = (
        out[[matricule_col, org_col, collect_col, ps_col]]
        .sort_values([matricule_col, org_col, collect_col], ascending=[True, True, True])
        .groupby([matricule_col, org_col], dropna=False, as_index=False)
        .first()   # conserve la ligne à plus vieille collecte
        .rename(columns={ps_col: "_PS_SUR_MINCOLLECTE", collect_col: "_MIN_COLLECTE"})
    )

    # 2) fallback si PS manquante sur la ligne min_collecte
    if fallback_when_ps_missing == "min_ps_non_null":
        # chercher la plus vieille PS non nulle du couple
        min_ps = (
            out[[matricule_col, org_col, ps_col]]
            .dropna(subset=[ps_col])
            .sort_values([matricule_col, org_col, ps_col])
            .groupby([matricule_col, org_col], dropna=False, as_index=False)
            .first()
            .rename(columns={ps_col: "_MIN_PS_NON_NULL"})
        )
        base = base.merge(min_ps, on=[matricule_col, org_col], how="left")

        # règle : PS = _PS_SUR_MINCOLLECTE si dispo, sinon _MIN_PS_NON_NULL, sinon _MIN_COLLECTE
        base[entry_col] = np.where(
            base["_PS_SUR_MINCOLLECTE"].notna(),
            base["_PS_SUR_MINCOLLECTE"],
            np.where(base["_MIN_PS_NON_NULL"].notna(), base["_MIN_PS_NON_NULL"], base["_MIN_COLLECTE"])
        )
    else:
        # règle par défaut : PS = _PS_SUR_MINCOLLECTE si dispo, sinon min collecte
        base[entry_col] = np.where(
            base["_PS_SUR_MINCOLLECTE"].notna(),
            base["_PS_SUR_MINCOLLECTE"],
            base["_MIN_COLLECTE"]
        )

    base = base[[matricule_col, org_col, entry_col]]

    # 3) joindre l'entrée au panel
    out = out.drop(columns=[entry_col], errors="ignore")
    out = out.merge(base, on=[matricule_col, org_col], how="left")
    out[entry_col] = pd.to_datetime(out[entry_col], errors="coerce")

    # 4) calcul des mois inclusifs
    def _months_diff_inclusive(d0, d1, inclusive=True):
        if pd.isna(d0) or pd.isna(d1):
            return np.nan
        m0 = pd.Timestamp(d0).to_period("M")
        m1 = pd.Timestamp(d1).to_period("M")
        if m1 < m0:
            return 0
        base = (m1.year - m0.year) * 12 + (m1.month - m0.month)
        return base + 1 if inclusive else base

    out[months_col] = [
        _months_diff_inclusive(de, pe, inclusive) for de, pe in zip(out[entry_col], out["PERIODE"])
    ]
    out[months_col] = pd.to_numeric(out[months_col], errors="coerce").astype("Int64")

    # 5) années
    out[years_col] = (out[months_col] // 12).astype("Int64")

    if return_report:
        rep = {
            "nb_couples_personne_organisme": int(base.shape[0]),
            "extrait": out[[matricule_col, org_col, "PERIODE", entry_col, months_col, years_col]]
                        .sort_values([matricule_col, org_col, "PERIODE"])
                        .head(20)
        }
        return out, rep

    return out


## CONTROLE 

#### Fonctions communs 

🟦 Variables communes (toutes catégories)

DATE_REF : date mensuelle de référence (début de mois) reconstruite depuis DATE_COLLECTE, ou via ANNEE/MOIS, PERIODE, ou DATE.

_YEAR_DELTA : écart en années entre deux observations successives d’un même matricule.

Colonnes techniques _... : mémorisent la colonne réellement utilisée dans les tests (_AGE_COL, _ENF_COL, _AGE_PS_COL, _ANC_COL, etc.).

In [37]:
# ============================================================
# HELPERS (communs)
# ============================================================
import pandas as pd
import numpy as np
import unicodedata, re

# --- normalisation texte basique
def _norm_txt(x):
    if pd.isna(x): return x
    x = str(x).strip().lower()
    x = ''.join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))
    x = re.sub(r"[-_'/\.]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# --- normalisation sexe -> {M,F,NaN}
def _sex_norm(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.upper().replace({
        'MASCULIN':'M','HOMME':'M','H':'M',
        'FEMININ':'F','FEMME':'F'
    })
    s = s.where(s.isin(['M','F']))
    return s

# --- construction de DATE_REF (début de mois) avec priorité à DATE_COLLECTE
def _ensure_date_ref(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'DATE_COLLECTE' in df.columns:
        s = df['DATE_COLLECTE']
        parsed_std = pd.to_datetime(s, errors='coerce')
        parsed_final = parsed_std
        try:
            needs_excel = parsed_std.isna().mean() > 0.5 and pd.api.types.is_numeric_dtype(s)
        except Exception:
            needs_excel = False
        if needs_excel:
            parsed_excel = pd.to_datetime(s, unit='D', origin='1899-12-30', errors='coerce')
            if parsed_excel.notna().sum() > parsed_std.notna().sum():
                parsed_final = parsed_excel
        df['DATE_REF'] = parsed_final.dt.to_period('M').dt.to_timestamp(how='start')
        return df
    if 'DATE_REF' in df.columns:
        df['DATE_REF'] = pd.to_datetime(df['DATE_REF'], errors='coerce').dt.to_period('M').dt.to_timestamp(how='start')
        return df
    # couples Année/Mois
    Yc, Mc = ['ANNEE_COLLECTE','ANNEE','YEAR','AN'], ['MOIS_COLLECTE','MOIS','MONTH']
    Y = next((c for c in Yc if c in df.columns), None)
    M = next((c for c in Mc if c in df.columns), None)
    if Y is not None:
        y = pd.to_numeric(df[Y], errors='coerce').round(0).astype('Int64').astype(str)
        m = (pd.to_numeric(df[M], errors='coerce') if M in df.columns else 1)
        m = pd.Series(m, index=df.index).round(0).astype('Int64').astype(str).str.zfill(2)
        df['DATE_REF'] = pd.to_datetime(y+'-'+m+'-01', format='%Y-%m-%d', errors='coerce')
        return df
    # PERIODE (YYYYMM / YYYY-MM / YYYY/MM)
    if 'PERIODE' in df.columns:
        per = df['PERIODE'].astype(str).str.strip()
        per_digits = per.str.replace(r'[^0-9]', '', regex=True)
        date_ref = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
        m6 = per_digits.str.len()==6
        if m6.any(): date_ref.loc[m6] = pd.to_datetime(per_digits[m6], format='%Y%m', errors='coerce')
        per_dash = per.str.replace('/', '-', regex=False)
        m7 = per_dash.str.match(r'^\d{4}-\d{2}$', na=False)
        if m7.any(): date_ref.loc[m7] = pd.to_datetime(per_dash[m7], format='%Y-%m', errors='coerce')
        df['DATE_REF'] = date_ref.dt.to_period('M').dt.to_timestamp(how='start')
        return df
    # DATE générique
    if 'DATE' in df.columns:
        df['DATE_REF'] = pd.to_datetime(df['DATE'], errors='coerce').dt.to_period('M').dt.to_timestamp(how='start')
        return df
    raise ValueError("DATE_REF introuvable : fournis DATE_COLLECTE ou un fallback (DATE_REF / ANNEE+MOIS / PERIODE / DATE).")

# --- vérifs clés temporelles
def _need_keys(df):
    if 'MATRICULE' not in df.columns: raise ValueError("Colonne MATRICULE manquante.")
    if 'DATE_REF' not in df.columns:  raise ValueError("Colonne DATE_REF manquante (utilise _ensure_date_ref).")

# --- delta temps (années) dans un groupe
def _add_year_delta(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('DATE_REF').copy()
    g['_DATE_PREV']  = g['DATE_REF'].shift(1)
    g['_YEAR_DELTA'] = ((g['DATE_REF'] - g['_DATE_PREV']).dt.days / 365.25).fillna(0)
    return g

# --- helper groupby.apply compatible
def _gb_apply(df: pd.DataFrame, by, func):
    gb = df.groupby(by, group_keys=False)
    try:
        return gb.apply(func, include_groups=False)   # pandas >= 2.2
    except TypeError:
        return gb.apply(func)

# ============================================================
# FLAGS DEMANDÉS — FONCTIONS DE CALCUL
# ============================================================

# ---------- 1) SITUATION MATRIMONIALE ----------
def sitmat_flags(df: pd.DataFrame, age_col=None, enfants_col=None) -> pd.DataFrame:
    df = df.copy()
    # colonnes
    src = 'SITUATION MATRIMONIALE' if 'SITUATION MATRIMONIALE' in df.columns else (
          'SITUATION_MATRIMONIALE' if 'SITUATION_MATRIMONIALE' in df.columns else None)
    if src is None:
        df['SITMAT_TMP'] = np.nan; src = 'SITMAT_TMP'
    if age_col is None:
        age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
        if age_col is None: df['AGE_TMP'] = np.nan; age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
    if enfants_col is None:
        enfants_col = 'NOMBRE ENFANT' if 'NOMBRE ENFANT' in df.columns else ('NB_ENFANTS' if 'NB_ENFANTS' in df.columns else None)
        if enfants_col is None: df['NB_ENFANTS_TMP'] = np.nan; enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    # recodage simple
    df['_SITMAT_NORM'] = df[src].map(_norm_txt)
    map_sit = {
        'celibataire':'celibataire','celib':'celibataire',
        'marie':'marie(e)','mariee':'marie(e)','femme mariee':'marie(e)','invalide marie':'marie(e)',
        'union libre':'union libre','concubinage':'union libre','concubin':'union libre',
        'divorce':'divorce/separe','divorce separe':'divorce/separe','separe':'divorce/separe',
        'veuf':'veuf/veuve','veuve':'veuf/veuve',
        'cas particulier':'autres','autre':'autres','autres':'autres'
    }
    df['_SITMAT_RECODE'] = df['_SITMAT_NORM'].replace(map_sit).fillna('autres')

    # flags demandés
    df['FLAG_SITMAT_AGE']        = ((df['_SITMAT_RECODE'].eq('marie(e)')) & (df[age_col] < 18)).astype('Int8')
    # temporel (variations d’un mois à l’autre + >1/an)
    if 'DATE_REF' not in df.columns: df = _ensure_date_ref(df)
    _need_keys(df)
    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        prev = g['_SITMAT_RECODE'].shift(1)
        g['FLAG_SITMAT_VARIATION'] = ((g['_SITMAT_RECODE'] != prev) &
                                      g['_SITMAT_RECODE'].notna() & prev.notna()).astype('Int8')
        g['_YEAR'] = g['DATE_REF'].dt.year
        changes_per_year = g['FLAG_SITMAT_VARIATION'].groupby(g['_YEAR']).transform('sum').fillna(0)
        g['FLAG_SITMAT_VARIATIONS'] = (changes_per_year >= 2).astype('Int8')
        return g.drop(columns=['_YEAR'], errors='ignore')
    df = _gb_apply(df, by='MATRICULE', func=_per_person)
    return df

# ---------- 2) ENFANTS ----------
def enfants_flags(df: pd.DataFrame, age_col=None) -> pd.DataFrame:
    df = df.copy()
    enfants_col = 'NOMBRE ENFANT' if 'NOMBRE ENFANT' in df.columns else ('NB_ENFANTS' if 'NB_ENFANTS' in df.columns else None)
    if enfants_col is None: df['NB_ENFANTS_TMP'] = np.nan; enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    if age_col is None:
        age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
        if age_col is None: df['AGE_TMP'] = np.nan; age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    # flags demandés
    df['FLAG_ENFANTS_AGE']      = ((df[age_col] < 15) & (df[enfants_col] > 0)).astype('Int8')
    df['FLAG_ENFANTS_FERT_IRR'] = (df[enfants_col] > (df[age_col] - 12)).astype('Int8')
    # temporel : descente du nb d'enfants
    if 'DATE_REF' not in df.columns: df = _ensure_date_ref(df)
    _need_keys(df)
    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        prev = g[enfants_col].shift(1)
        g['FLAG_ENFANTS_DESCENTE'] = ((g[enfants_col] < prev) & g[enfants_col].notna() & prev.notna()).astype('Int8')
        return g
    df = _gb_apply(df, by='MATRICULE', func=_per_person)
    return df

# ---------- 3) SEXE & ÂGE (cohérence temporelle) ----------
def sexe_age_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # normaliser sexe
    sex_src = 'SEXE' if 'SEXE' in df.columns else None
    if sex_src is None: df['SEXE_TMP'] = np.nan; sex_src = 'SEXE_TMP'
    df['_SEXE_NORM'] = _sex_norm(df[sex_src])

    # âge
    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    if age_col is None: df['AGE_TMP']=np.nan; age_col='AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    if 'DATE_REF' not in df.columns: df = _ensure_date_ref(df)
    _need_keys(df)

    def _per_person(g):
        g = _add_year_delta(g)
        # variation de sexe
        g['_SEXE_PREV'] = g['_SEXE_NORM'].shift(1)
        g['FLAG_SEXE_VARIATION'] = ((g['_SEXE_NORM'] != g['_SEXE_PREV']) &
                                    g['_SEXE_NORM'].notna() & g['_SEXE_PREV'].notna()).astype('Int8')
        # âge décroît / trop rapide (> +3 mois/an)
        g['_AGE_PREV'] = g[age_col].shift(1)
        g['FLAG_AGE_DECROIT']     = ((g[age_col] < g['_AGE_PREV']) & g[age_col].notna() & g['_AGE_PREV'].notna()).astype('Int8')
        g['FLAG_AGE_TROP_RAPIDE'] = (((g[age_col]-g['_AGE_PREV']) > (g['_YEAR_DELTA'] + 0.25)) &
                                     g[age_col].notna() & g['_AGE_PREV'].notna()).astype('Int8')
        return g.drop(columns=['_SEXE_PREV','_AGE_PREV'], errors='ignore')
    return _gb_apply(df, by='MATRICULE', func=_per_person)

# ---------- 4) ÂGE & PRISE DE SERVICE ----------
def age_ps_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # AGE_PS_VARIATION sur une colonne d'âge de prise de service si dispo
    ps_col = 'AGE_PRISERVICE'
    if ps_col not in df.columns:
        # fallback possible à partir de dates si tu les as (désactivé par défaut)
        df[ps_col] = np.nan
    df[ps_col] = pd.to_numeric(df[ps_col], errors='coerce')

    # AGE_PS_GE_AGE
    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    if age_col:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        df['FLAG_AGE_PS_GE_AGE'] = ((df[ps_col] >= df[age_col]) & df[ps_col].notna() & df[age_col].notna()).astype('Int8')
    else:
        df['FLAG_AGE_PS_GE_AGE'] = pd.Series(0, index=df.index, dtype='Int8')

    # variation temporelle de l'âge de prise de service
    if 'DATE_REF' not in df.columns: df = _ensure_date_ref(df)
    _need_keys(df)
    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        prev = g[ps_col].shift(1)
        g['FLAG_AGE_PS_VARIATION'] = ((g[ps_col] != prev) & g[ps_col].notna() & prev.notna()).astype('Int8')
        return g
    return _gb_apply(df, by='MATRICULE', func=_per_person)

# ---------- 5) ANCIENNETÉ ----------
def anciennete_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    anc_col = 'ANCIENNETE' if 'ANCIENNETE' in df.columns else None
    if anc_col is None: df['ANCIENNETE']=np.nan; anc_col='ANCIENNETE'
    df[anc_col] = pd.to_numeric(df[anc_col], errors='coerce')

    # ANC_NEGATIVE
    df['FLAG_ANC_NEGATIVE'] = (df[anc_col] < 0).astype('Int8')

    # ANC_IDENTITE_IRR : |AGE - (AGE_PS + ANC)| > 1
    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    ps_col  = 'AGE_PRISERVICE' if 'AGE_PRISERVICE' in df.columns else None
    if age_col and ps_col:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        df[ps_col]  = pd.to_numeric(df[ps_col], errors='coerce')
        df['FLAG_ANC_IDENTITE_IRR'] = (np.abs(df[age_col] - (df[ps_col] + df[anc_col])) > 1.0).astype('Int8')
    else:
        df['FLAG_ANC_IDENTITE_IRR'] = pd.Series(0, index=df.index, dtype='Int8')

    # temporel : baisse & vitesse > +3 mois/an
    if 'DATE_REF' not in df.columns: df = _ensure_date_ref(df)
    _need_keys(df)
    def _per_person(g):
        g = _add_year_delta(g)
        prev = g[anc_col].shift(1)
        g['FLAG_ANC_BAISSE']       = ((g[anc_col] < prev) & g[anc_col].notna() & prev.notna()).astype('Int8')
        g['FLAG_ANC_VITESSE_IRR']  = (((g[anc_col]-prev) > (g['_YEAR_DELTA'] + 0.25)) &
                                      g[anc_col].notna() & prev.notna()).astype('Int8')
        return g
    return _gb_apply(df, by='MATRICULE', func=_per_person)

# ============================================================
# PIPELINE : APPLIQUER UNIQUEMENT TES FLAGS + EXPORT
# ============================================================
FLAG_MAP = {
    "Situation_matrimoniale": [
        "FLAG_SITMAT_AGE", "FLAG_SITMAT_VARIATION", "FLAG_SITMAT_VARIATIONS"
    ],
    "Enfants_fertilite": [
        "FLAG_ENFANTS_AGE", "FLAG_ENFANTS_FERT_IRR", "FLAG_ENFANTS_DESCENTE"
    ],
    "Sexe_Age_Cohérence": [
        "FLAG_SEXE_VARIATION", "FLAG_AGE_DECROIT", "FLAG_AGE_TROP_RAPIDE"
    ],
    "Age_&_Prise_de_service": [
        "FLAG_AGE_PS_VARIATION", "FLAG_AGE_PS_GE_AGE"
    ],
    "Anciennete": [
        "FLAG_ANC_NEGATIVE", "FLAG_ANC_IDENTITE_IRR", "FLAG_ANC_BAISSE", "FLAG_ANC_VITESSE_IRR"
    ],
}

def compute_selected_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Applique uniquement les flags sélectionnés ci-dessus."""
    out = df.copy()
    out = _ensure_date_ref(out)
    _need_keys(out)
    out = sitmat_flags(out)
    out = enfants_flags(out)
    out = sexe_age_flags(out)
    out = age_ps_flags(out)
    out = anciennete_flags(out)
    return out

def export_selected_flags_report(
    df: pd.DataFrame,
    outfile: str = "rapport_flags_selection.xlsx",
    key_cols=("MATRICULE",),
    date_cols=("DATE_REF","DATE_COLLECTE"),
    extra_cols=("AGE","AGE_IMPUTE","AGE_VALIDE","SEXE","SEXE_IMPUTE","AGE_PRISERVICE","ANCIENNETE"),
    flag_map: dict = None,
    flag_value_active=(1, True, "1", "TRUE")
):
    if flag_map is None:
        flag_map = FLAG_MAP
    df = df.copy()
    # colonnes utiles
    base_cols = [c for c in list(key_cols)+list(date_cols)+list(extra_cols) if c in df.columns]
    # liste finale des flags à considérer
    all_flags = sorted({f for lst in flag_map.values() for f in lst if f in df.columns})
    if not all_flags:
        raise ValueError("Aucun flag sélectionné trouvé dans le DataFrame.")
    # binaire
    def _to_active(x):
        if pd.isna(x): return 0
        if isinstance(x, str):
            xu = x.strip().upper()
            return 1 if xu in {str(v).upper() for v in flag_value_active} else 0
        try:
            return 1 if x in flag_value_active else int(bool(x))
        except Exception:
            return int(bool(x))
    flags_bin = df[all_flags].applymap(_to_active)
    n_total = len(df)

    # KPI par flag
    kpi = (flags_bin.sum(axis=0)
           .rename("Effectif").to_frame()
           .assign(Taux_pct=lambda t: (t["Effectif"]/n_total*100).round(2))
           .sort_values("Effectif", ascending=False))
    kpi.index.name = "FLAG"

    # Matricules à problème
    df["_NB_FLAGS_LIGNE"] = flags_bin.sum(axis=1)
    group_keys = [c for c in key_cols if c in df.columns]
    if not group_keys:
        df["ROW_ID"] = np.arange(len(df))+1
        group_keys = ["ROW_ID"]
    agg_prob = (df.assign(_ANY_FLAG=(df["_NB_FLAGS_LIGNE"]>0).astype(int))
                  .groupby(group_keys, as_index=False)
                  .agg(NB_LIGNES_FLAGGEES=("_ANY_FLAG","sum"),
                       NB_FLAGS_TOTAL=("_NB_FLAGS_LIGNE","sum"))
                  .sort_values(["NB_FLAGS_TOTAL","NB_LIGNES_FLAGGEES"], ascending=False))

    # Export
    with pd.ExcelWriter(outfile, engine="xlsxwriter") as xw:
        kpi.to_excel(xw, sheet_name="KPI_flags")
        agg_prob.to_excel(xw, sheet_name="Matricules_problèmes", index=False)

        for cat, flist in flag_map.items():
            fl_real = [f for f in flist if f in df.columns]
            if not fl_real: continue
            mask_cat = (df[fl_real].applymap(_to_active).sum(axis=1) > 0)
            extrait = df.loc[mask_cat, base_cols + fl_real].copy()
            extrait["_NB_FLAGS_CATEG"] = df.loc[mask_cat, fl_real].applymap(_to_active).sum(axis=1).values
            sort_cols = [c for c in key_cols if c in extrait.columns] + ["_NB_FLAGS_CATEG"]
            extrait = extrait.sort_values(sort_cols, ascending=[True]*(len(sort_cols)-1)+[False])
            # ordonner flags par fréquence
            flags_sorted = sorted(fl_real, key=lambda c: extrait[c].apply(_to_active).sum(), reverse=True)
            ordered_cols = [c for c in base_cols if c in extrait.columns] + flags_sorted + ["_NB_FLAGS_CATEG"]
            extrait = extrait[ordered_cols]
            extrait.to_excel(xw, sheet_name=cat[:31], index=False)

        resume = pd.DataFrame({
            "Total_lignes":[n_total],
            "Total_flags_distincts":[len(all_flags)],
            "Total_matricules":[df[group_keys].drop_duplicates().shape[0]],
            "Lignes_avec_≥1_flag":[int((df['_NB_FLAGS_LIGNE']>0).sum())]
        })
        resume.to_excel(xw, sheet_name="Résumé", index=False)

    df.drop(columns=["_NB_FLAGS_LIGNE"], inplace=True, errors="ignore")
    print(f"✅ Rapport créé : {outfile}")




#### Situation Matrimoniale 

In [16]:
# ============================================================
# CATÉGORIE : SITUATION MATRIMONIALE
# ============================================================
# --- cohérence non temporelle
def sitmat_coherence_non_temporelle(df: pd.DataFrame, age_col: str = None, enfants_col: str = None) -> pd.DataFrame:
    df = df.copy()
    src = 'SITUATION MATRIMONIALE' if 'SITUATION MATRIMONIALE' in df.columns else (
          'SITUATION_MATRIMONIALE' if 'SITUATION_MATRIMONIALE' in df.columns else None)
    if src is None:
        df['SITMAT_TMP'] = np.nan
        src = 'SITMAT_TMP'

    if age_col is None:
        age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
        if age_col is None: df['AGE_TMP'] = np.nan; age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    if enfants_col is None:
        enfants_col = 'NOMBRE ENFANT' if 'NOMBRE ENFANT' in df.columns else ('NB_ENFANTS' if 'NB_ENFANTS' in df.columns else None)
        if enfants_col is None: df['NB_ENFANTS_TMP'] = np.nan; enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    df['_SITMAT_NORM'] = df[src].map(_norm_txt)
    map_sit = {
        'celibataire':'celibataire','celib':'celibataire',
        'marie':'marie(e)','mariee':'marie(e)','femme mariee':'marie(e)','invalide marie':'marie(e)',
        'union libre':'union libre','concubinage':'union libre','concubin':'union libre',
        'divorce':'divorce/separe','divorce separe':'divorce/separe','separe':'divorce/separe',
        'veuf':'veuf/veuve','veuve':'veuf/veuve',
        'cas particulier':'autres','autre':'autres','autres':'autres'
    }
    df['_SITMAT_RECODE'] = df['_SITMAT_NORM'].replace(map_sit)

    df['FLAG_SITMAT_INCONNU']    = df['_SITMAT_NORM'].isna().astype('Int8')
    df['FLAG_SITMAT_NON_MAPPEE'] = ((~df['_SITMAT_NORM'].isna()) & (df['_SITMAT_RECODE'].isna())).astype('Int8')
    df['_SITMAT_RECODE'] = df['_SITMAT_RECODE'].fillna('autres')

    df['FLAG_SITMAT_ENFANTS'] = ((df['_SITMAT_RECODE'] == 'celibataire') & (df[enfants_col] > 0)).astype('Int8')
    df['FLAG_SITMAT_AGE']     = ((df['_SITMAT_RECODE'].isin(['marie(e)'])) & (df[age_col] < 18)).astype('Int8')

    df['_AGE_COL_FOR_SITMAT'] = age_col
    df['_ENF_COL_FOR_SITMAT'] = enfants_col
    return df

# --- cohérence temporelle
def sitmat_coherence_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    if '_SITMAT_RECODE' not in df.columns:
        raise ValueError("La colonne _SITMAT_RECODE est manquante (appelle sitmat_coherence_non_temporelle d'abord).")

    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        prev = g['_SITMAT_RECODE'].shift(1)
        g['FLAG_SITMAT_VARIATION'] = ((g['_SITMAT_RECODE'] != prev) &
                                      g['_SITMAT_RECODE'].notna() & prev.notna()).astype('Int8')
        g['_YEAR'] = g['DATE_REF'].dt.year
        changes_per_year = (g['FLAG_SITMAT_VARIATION'].groupby(g['_YEAR']).transform('sum').fillna(0))
        g['FLAG_SITMAT_VARIATIONS'] = (changes_per_year >= 2).astype('Int8')
        return g.drop(columns=['_YEAR'], errors='ignore')

    return df.groupby('MATRICULE', group_keys=False).apply(_per_person)


#### Nombre d'enfants

In [17]:

# ============================================================
# CATÉGORIE : NOMBRE D'ENFANTS
# ============================================================

# --- cohérence non temporelle
def enfants_coherence_non_temporelle(df: pd.DataFrame, age_col: str = None,
                                     max_enfants: int = 12, enfants_impute_col: str = None) -> pd.DataFrame:
    df = df.copy()
    enfants_col = 'NOMBRE ENFANT' if 'NOMBRE ENFANT' in df.columns else ('NB_ENFANTS' if 'NB_ENFANTS' in df.columns else None)
    if enfants_col is None: df['NB_ENFANTS_TMP'] = np.nan; enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    if age_col is None:
        age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
        if age_col is None: df['AGE_TMP'] = np.nan; age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    df['FLAG_ENFANTS_NEGATIF']  = (df[enfants_col] < 0).astype('Int8')
    df['FLAG_ENFANTS_OUTLIER']  = (df[enfants_col] > max_enfants).astype('Int8')
    df['FLAG_ENFANTS_MANQUANT'] = df[enfants_col].isna().astype('Int8')

    if enfants_impute_col and (enfants_impute_col in df.columns):
        df['FLAG_ENFANTS_IMPUTE'] = df[enfants_impute_col].astype('Int8')
    elif 'ENFANTS_IMPUTE' in df.columns:
        df['FLAG_ENFANTS_IMPUTE'] = df['ENFANTS_IMPUTE'].astype('Int8')
    else:
        df['FLAG_ENFANTS_IMPUTE'] = pd.Series(0, index=df.index, dtype='Int8')

    df['FLAG_ENFANTS_AGE']      = ((df[age_col] < 15) & (df[enfants_col] > 0)).astype('Int8')
    df['FLAG_ENFANTS_FERT_IRR'] = (df[enfants_col] > (df[age_col] - 12)).astype('Int8')

    df['_ENF_COL'] = enfants_col
    return df

# --- cohérence temporelle
def enfants_coherence_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    if '_ENF_COL' not in df.columns:
        raise ValueError("La colonne _ENF_COL est manquante (appelle enfants_coherence_non_temporelle d'abord).")
    df = df.copy()
    enfants_col = df['_ENF_COL'].iloc[0]

    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        prev = g[enfants_col].shift(1)
        g['FLAG_ENFANTS_DESCENTE'] = ((g[enfants_col] < prev) & g[enfants_col].notna() & prev.notna()).astype('Int8')
        return g

    return df.groupby('MATRICULE', group_keys=False).apply(_per_person)


#### Statut 

In [18]:

# ============================================================
# CATÉGORIE : STATUT
# ============================================================

# --- cohérence non temporelle
def statut_coherence_non_temporelle(df: pd.DataFrame, statut_ok=None, map_statut_to_def=None) -> pd.DataFrame:
    df = df.copy()
    statut_ok = set(statut_ok or {
        'en activite','disponibilite','detachement','suspension','fin de contrat',
        "fin d'engagement",'retraite','retraite pour limite age','retraite pour anciennete',
        'deces','arret recensement 2011','abandon de poste','regul. indemnites',
        'sous controle // traitement','arret police','viol'
    })
    map_statut_to_def = map_statut_to_def or {
        'en activite':'fonctionnaire','detachement':'fonctionnaire','disponibilite':'fonctionnaire',
        'suspension':'fonctionnaire','retraite':'fonctionnaire','retraite pour limite age':'fonctionnaire',
        'retraite pour anciennete':'fonctionnaire','fin de contrat':'non-fonctionnaire'
    }

    s_col  = 'STATUT' if 'STATUT' in df.columns else None
    if s_col is None:  df['STATUT_TMP'] = np.nan; s_col  = 'STATUT_TMP'
    sd_col = 'STATUT_DEFINITIF' if 'STATUT_DEFINITIF' in df.columns else None
    if sd_col is None: df['STATUT_DEFINITIF_TMP'] = np.nan; sd_col = 'STATUT_DEFINITIF_TMP'

    df['_STATUT_NORM']     = df[s_col].map(_norm_txt)
    df['_STATUT_DEF_NORM'] = df[sd_col].map(_norm_txt)

    df['FLAG_STATUT_NON_MAPPE'] = (~df['_STATUT_NORM'].isin(statut_ok) & df['_STATUT_NORM'].notna()).astype('Int8')

    expected_def = df['_STATUT_NORM'].map(map_statut_to_def).fillna('cas particulier')
    df['FLAG_STATUT_DEF_INCOHERENT'] = ((df['_STATUT_DEF_NORM'].notna()) & (df['_STATUT_DEF_NORM'] != expected_def)).astype('Int8')
    df['FLAG_STATUT_DEF_INCONNU']    = df['_STATUT_DEF_NORM'].isna().astype('Int8')

    # Retraite incohérente avec âge si dispo
    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    if age_col:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        _stat_s = df['_STATUT_NORM'].astype('string')
        df['FLAG_AGE_RETRAITE_IRR'] = (_stat_s.str.contains('retraite', na=False) &
                                       (df[age_col] < CFG['retraite_min_age'])).astype('Int8')
    else:
        df['FLAG_AGE_RETRAITE_IRR'] = pd.Series(0, index=df.index, dtype='Int8')
    return df

# --- cohérence temporelle
def statut_coherence_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    def _per_person(g):
        g = _add_year_delta(g)
        g['_STAT_PREV']   = g['_STATUT_NORM'].shift(1)
        g['_STATDEF_PREV']= g['_STATUT_DEF_NORM'].shift(1)
        g['FLAG_STATUT_VARIATION']     = ((g['_STATUT_NORM']     != g['_STAT_PREV'])    &
                                          g['_STATUT_NORM'].notna() & g['_STAT_PREV'].notna()).astype('Int8')
        g['FLAG_STATUT_DEF_VARIATION'] = ((g['_STATUT_DEF_NORM'] != g['_STATDEF_PREV']) &
                                          g['_STATUT_DEF_NORM'].notna() & g['_STATDEF_PREV'].notna()).astype('Int8')
        return g.drop(columns=['_STAT_PREV','_STATDEF_PREV'], errors='ignore')
    out = df.groupby('MATRICULE', group_keys=False).apply(_per_person)

    if {'ANNEE_COLLECTE','MOIS_COLLECTE'}.issubset(out.columns):
        key = ['MATRICULE','ANNEE_COLLECTE','MOIS_COLLECTE']
        nunq = (out.assign(_STAT_UNIQ=out['_STATUT_NORM'].fillna(''))
                   .groupby(key)['_STAT_UNIQ'].nunique())
        out = out.merge(nunq.rename('N_STATUTS'), left_on=key, right_index=True, how='left')
        out['FLAG_STATUT_MULTI'] = (out['N_STATUTS'] > 1).fillna(0).astype('Int8')
        has_stat = out['_STATUT_NORM'].notna().astype(int)
        any_by_key = has_stat.groupby([out[c] for c in key]).transform('max')
        out['FLAG_STATUT_TROU'] = (any_by_key == 0).astype('Int8')
        out.drop(columns=['N_STATUTS'], errors='ignore', inplace=True)
    else:
        out['FLAG_STATUT_MULTI'] = pd.Series(0, index=out.index, dtype='Int8')
        out['FLAG_STATUT_TROU']  = pd.Series(0, index=out.index, dtype='Int8')
    return out


### Sexe

In [19]:
# ============================================================
# CATÉGORIE : SEXE
# ============================================================

# --- cohérence non temporelle
def sexe_coherence_non_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    src = 'SEXE' if 'SEXE' in df.columns else None
    if src is None: df['SEXE_TMP'] = np.nan; src = 'SEXE_TMP'
    df['_SEXE_NORM'] = _sex_norm(df[src])
    df['FLAG_SEXE_INVALIDE'] = (~df['_SEXE_NORM'].isin(['M','F'])).fillna(True).astype('Int8')
    df['FLAG_SEXE_INCONNU']  = df['_SEXE_NORM'].isna().astype('Int8')
    return df

# --- cohérence temporelle
def sexe_coherence_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    if '_SEXE_NORM' not in df.columns: raise ValueError("Normalise d’abord le sexe (M/F).")
    df = df.copy()
    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        g['_SEXE_PREV'] = g['_SEXE_NORM'].shift(1)
        g['FLAG_SEXE_VARIATION'] = ((g['_SEXE_NORM'] != g['_SEXE_PREV']) &
                                    g['_SEXE_NORM'].notna() & g['_SEXE_PREV'].notna()).astype('Int8')
        return g.drop(columns=['_SEXE_PREV'], errors='ignore')
    return df.groupby('MATRICULE', group_keys=False).apply(_per_person)


### Age

In [20]:
# ============================================================
# CATÉGORIE : ÂGE
# ============================================================

# --- cohérence non temporelle
def age_coherence_non_temporelle(df: pd.DataFrame, bounds=(18,75)) -> pd.DataFrame:
    df = df.copy()
    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    if age_col is None: df['AGE_TMP']=np.nan; age_col='AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
    amin, amax = bounds
    df['FLAG_AGE_OUTLIER'] = ((df[age_col] < amin) | (df[age_col] > amax)).astype('Int8')
    df['_AGE_COL'] = age_col
    return df

# --- cohérence temporelle
def age_coherence_temporelle(df: pd.DataFrame, max_drift=0.25) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    if '_AGE_COL' not in df.columns: raise ValueError("Appelle d’abord age_coherence_non_temporelle pour définir _AGE_COL.")
    df = df.copy()
    age_col = df['_AGE_COL'].iloc[0]
    def _per_person(g):
        g = _add_year_delta(g)
        g['_AGE_PREV'] = g[age_col].shift(1)
        g['FLAG_AGE_DECROIT']      = ((g[age_col] < g['_AGE_PREV']) & g[age_col].notna() & g['_AGE_PREV'].notna()).astype('Int8')
        g['FLAG_AGE_TROP_RAPIDE']  = (((g[age_col]-g['_AGE_PREV']) > (g['_YEAR_DELTA']+max_drift)) &
                                      g[age_col].notna() & g['_AGE_PREV'].notna()).astype('Int8')
        return g.drop(columns=['_AGE_PREV'], errors='ignore')
    return df.groupby('MATRICULE', group_keys=False).apply(_per_person)


### Age de prise de service 

In [21]:
# ============================================================
# CATÉGORIE : ÂGE DE PRISE DE SERVICE
# ============================================================

# --- cohérence non temporelle
def ageps_coherence_non_temporelle(df: pd.DataFrame, bounds=(14,60)) -> pd.DataFrame:
    df = df.copy()
    ps = 'AGE_PRISERVICE' if 'AGE_PRISERVICE' in df.columns else None
    if ps is None:
        if {'DATE_PRISERVICE','DATE_NAISSANCE'}.issubset(df.columns):
            dps = pd.to_datetime(df['DATE_PRISERVICE'], errors='coerce')
            dob = pd.to_datetime(df['DATE_NAISSANCE'], errors='coerce')
            df['AGE_PS_CALC'] = ((dps - dob).dt.days / 365.25)
            ps = 'AGE_PS_CALC'
        else:
            df['AGE_PS_TMP'] = np.nan; ps = 'AGE_PS_TMP'
    df[ps] = pd.to_numeric(df[ps], errors='coerce')
    amin, amax = bounds
    df['FLAG_AGE_PS_OUTLIER'] = ((df[ps] < amin) | (df[ps] > amax)).astype('Int8')

    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    if age_col:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        df['FLAG_AGE_PS_GE_AGE'] = ((df[ps] >= df[age_col]) & df[ps].notna() & df[age_col].notna()).astype('Int8')
    else:
        df['FLAG_AGE_PS_GE_AGE'] = pd.Series(0, index=df.index, dtype='Int8')

    df['_AGE_PS_COL'] = ps
    return df

# --- cohérence temporelle
def ageps_coherence_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    if '_AGE_PS_COL' not in df.columns: raise ValueError("Appelle d’abord ageps_coherence_non_temporelle pour définir _AGE_PS_COL.")
    df = df.copy()
    ps_col = df['_AGE_PS_COL'].iloc[0]
    def _per_person(g):
        g = g.sort_values('DATE_REF').copy()
        g['_AGEPS_PREV'] = g[ps_col].shift(1)
        g['FLAG_AGE_PS_VARIATION'] = ((g[ps_col] != g['_AGEPS_PREV']) &
                                      g[ps_col].notna() & g['_AGEPS_PREV'].notna()).astype('Int8')
        return g.drop(columns=['_AGEPS_PREV'], errors='ignore')
    return df.groupby('MATRICULE', group_keys=False).apply(_per_person)


# RAPPORT FLAGS → EXCEL

On génère un **rapport Excel de flags** à partir d’un DataFrame et d’une clé d’identification robuste.

**Clé de groupement (`key_cols`)**  
- Utilise la/les colonnes fournies si elles existent ; sinon **détection automatique** (variantes de *MATRICULE*).  
- Si aucune clé n’est trouvée, création d’une **colonne `ROW_ID`** séquentielle.

**KPI des flags (`KPI_flags`)**  
- Tableau récapitulatif par `FLAG` avec le **taux (%)** d’activation : `TAUX_%`.  
- Tri décroissant des flags les plus fréquents.

**Clés à problème (`Clés_problèmes`)**  
- Agrégation par clé : `NB_LIGNES_FLAGGEES` = nombre de lignes comportant ≥1 flag.  
- Permet d’identifier les **matricules/agents** les plus affectés.

**Lignes détaillées (`Lignes_détaillées`)**  
- Filtre des **lignes avec au moins un flag**.  
- Colonnes : **clé(s)** + `DATE_REF` (si présente) + `extra_cols` + `flag_cols`.  
- Tri par clé(s) puis `DATE_REF` (si disponible).

**Top valeurs non mappées (onglets conditionnels)**  
- `Top_sitmat_non_mappées` si `FLAG_SITMAT_NON_MAPPEE` et `_SITMAT_NORM` existent : **top 50** des valeurs non mappées.  
- `Top_statut_non_mappés` si `FLAG_STATUT_NON_MAPPE` et `_STATUT_NORM` existent : **top 50** des valeurs non mappées.

**Sortie**  
- Écriture d’un fichier **Excel** nommé à partir de `outfile` (suffixe `.xlsx`) avec les onglets ci-dessus.  
- Messages console : clé utilisée, taux de flags, total de lignes flaggées, top des clés à problème.


In [35]:
# ======================= RAPPORT FLAGS → EXCEL (robuste) =======================
import pandas as pd
from pathlib import Path
import re
import unicodedata

def _norm_name(s: str) -> str:
    s = ''.join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return s

def _auto_key_cols(df: pd.DataFrame, preferred:list) -> list:
    """Trouve une clé de groupement robuste (MATRICULE, variantes, etc.), sinon crée ROW_ID."""
    cols = list(df.columns)
    norm_map = {_norm_name(c): c for c in cols}
    # candidats usuels
    candidates = preferred or []
    candidates += ['MATRICULE','Matricule','matricule','ID','Id','id','ID_AGENT','IDAGENT','CODE_AGENT','CODEAGENT']
    # tenter correspondance normalisée
    for cand in candidates:
        nc = _norm_name(cand)
        if nc in norm_map:
            return [norm_map[nc]]
    # scan heuristique : colonnes qui ressemblent à "matricule"
    for c in cols:
        if re.search(r'matric', _norm_name(c)):
            return [c]
    # sinon fallback : ROW_ID
    if 'ROW_ID' not in df.columns:
        df['ROW_ID'] = range(1, len(df)+1)
    return ['ROW_ID']

def export_flags_report(
    df: pd.DataFrame,
    flag_cols: list,
    key_cols: list = None,
    extra_cols: list = None,
    outfile: str = "rapport_flags",
    title: str = "Rapport flags"
):
    df = df.copy()

    # 0) Sélection des flags présents
    flag_cols = [c for c in (flag_cols or []) if c in df.columns]
    if not flag_cols:
        raise ValueError("Aucun des flags spécifiés n'existe dans le DataFrame.")

    # 1) Trouver une clé de groupement robuste
    key_cols = key_cols or []
    # garder seulement celles présentes
    key_cols = [c for c in key_cols if c in df.columns]
    if not key_cols:
        key_cols = _auto_key_cols(df, preferred=['MATRICULE'])
    print(f"[INFO] Clé de groupement utilisée : {key_cols}")

    # 2) Colonnes extra
    extra_cols = [c for c in (extra_cols or []) if c in df.columns]

    # 3) KPI (taux en % par flag)
    kpi = (df[flag_cols].mean(numeric_only=True) * 100).round(2).sort_values(ascending=False)
    kpi_df = kpi.rename_axis("FLAG").reset_index(name="TAUX_%")

    # 4) Matricules/Clés à problème (≥1 flag)
    any_flag = (df[flag_cols].sum(axis=1) > 0)
    prob_df = (df.assign(_ANY=any_flag)
                 .groupby(key_cols, dropna=False)['_ANY']
                 .sum()
                 .reset_index(name='NB_LIGNES_FLAGGEES')
                 .sort_values('NB_LIGNES_FLAGGEES', ascending=False))

    # 5) Lignes détaillées (toutes les lignes avec ≥1 flag)
    view_cols = list(dict.fromkeys(key_cols + (['DATE_REF'] if 'DATE_REF' in df.columns else []) + extra_cols + flag_cols))
    details_df = df.loc[any_flag, view_cols].sort_values(key_cols + (['DATE_REF'] if 'DATE_REF' in view_cols else []))

    # 6) Top valeurs non mappées (sitmat / statut)
    top_unmapped_df = pd.DataFrame()
    if 'FLAG_SITMAT_NON_MAPPEE' in flag_cols and '_SITMAT_NORM' in df.columns:
        top_unmapped_df = (df.loc[df['FLAG_SITMAT_NON_MAPPEE'] == 1, '_SITMAT_NORM']
                             .value_counts().head(50)
                             .rename_axis('_SITMAT_NORM').reset_index(name='NB'))
    statut_unmapped_df = pd.DataFrame()
    if 'FLAG_STATUT_NON_MAPPE' in flag_cols and '_STATUT_NORM' in df.columns:
        statut_unmapped_df = (df.loc[df['FLAG_STATUT_NON_MAPPE'] == 1, '_STATUT_NORM']
                                .value_counts().head(50)
                                .rename_axis('_STATUT_NORM').reset_index(name='NB'))

    # 7) Affichages console
    print(f"\n===== {title} =====")
    print("\nTaux de flags (%) :")
    print(kpi_df.to_string(index=False))
    print(f"\nTotal lignes avec ≥1 flag : {any_flag.sum()} / {len(df)} ({(100*any_flag.mean()):.2f}%)")
    print("\nTop clés à problème :")
    print(prob_df.head(20).to_string(index=False))

    # 8) Écriture Excel
    outfile = str(Path(outfile).with_suffix('.xlsx'))
    with pd.ExcelWriter(outfile, engine='openpyxl') as xw:
        kpi_df.to_excel(xw, index=False, sheet_name='KPI_flags')
        prob_df.to_excel(xw, index=False, sheet_name='Clés_problèmes')
        details_df.to_excel(xw, index=False, sheet_name='Lignes_détaillées')
        if not top_unmapped_df.empty:
            top_unmapped_df.to_excel(xw, index=False, sheet_name='Top_sitmat_non_mappées')
        if not statut_unmapped_df.empty:
            statut_unmapped_df.to_excel(xw, index=False, sheet_name='Top_statut_non_mappés')

    print(f"\n✅ Fichier Excel écrit : {outfile}")




# 2) (Optionnel) Nombre d’enfants — décommente si tu veux générer aussi
# enf_flag_cols = [
#     'FLAG_ENFANTS_NEGATIF','FLAG_ENFANTS_OUTLIER','FLAG_ENFANTS_MANQUANT',
#     'FLAG_ENFANTS_IMPUTE','FLAG_ENFANTS_AGE','FLAG_ENFANTS_FERT_IRR','FLAG_ENFANTS_DESCENTE'
# ]
# enf_key_cols = ['MATRICULE']
# enf_extra_cols = ['NB_ENFANTS','NOMBRE ENFANT','AGE','AGE_IMPUTE']
# export_flags_report(panel_solde_df, enf_flag_cols, enf_key_cols, enf_extra_cols,
#                     outfile='rapport_flags_enfants', title='NOMBRE D’ENFANTS')

# 3) (Optionnel) Statut — décommente si besoin
# statut_flag_cols = [
#     'FLAG_STATUT_NON_MAPPE','FLAG_STATUT_DEF_INCOHERENT','FLAG_STATUT_DEF_INCONNU',
#     'FLAG_AGE_RETRAITE_IRR','FLAG_STATUT_VARIATION','FLAG_STATUT_DEF_VARIATION',
#     'FLAG_STATUT_MULTI','FLAG_STATUT_TROU'
# ]
# statut_key_cols = ['MATRICULE']
# statut_extra_cols = ['STATUT','STATUT_DEFINITIF','_STATUT_NORM','_STATUT_DEF_NORM','AGE','AGE_IMPUTE']
# export_flags_report(panel_solde_df, statut_flag_cols, statut_key_cols, statut_extra_cols,
#                     outfile='rapport_flags_statut', title='STATUT')


### Ancienneté 

In [22]:

# ============================================================
# CATÉGORIE : ANCIENNETÉ
# ============================================================

# --- cohérence non temporelle
def anc_coherence_non_temporelle(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    anc = 'ANCIENNETE' if 'ANCIENNETE' in df.columns else None
    if anc is None: df['ANCIENNETE_TMP']=np.nan; anc='ANCIENNETE_TMP'
    df[anc] = pd.to_numeric(df[anc], errors='coerce')
    df['FLAG_ANC_NEGATIVE'] = (df[anc] < 0).astype('Int8')

    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    age_ps  = 'AGE_PRISERVICE' if 'AGE_PRISERVICE' in df.columns else ('AGE_PS_CALC' if 'AGE_PS_CALC' in df.columns else None)
    if age_col and age_ps:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        df[age_ps]  = pd.to_numeric(df[age_ps],  errors='coerce')
        df['FLAG_ANC_IDENTITE_IRR'] = (np.abs(df[age_col] - (df[age_ps] + df[anc])) > 1.0).astype('Int8')
    else:
        df['FLAG_ANC_IDENTITE_IRR'] = pd.Series(0, index=df.index, dtype='Int8')

    df['_ANC_COL'] = anc
    return df

# --- cohérence temporelle
def anc_coherence_temporelle(df: pd.DataFrame, max_drift=0.25) -> pd.DataFrame:
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    if '_ANC_COL' not in df.columns: raise ValueError("Appelle d’abord anc_coherence_non_temporelle pour définir _ANC_COL.")
    df = df.copy()
    anc = df['_ANC_COL'].iloc[0]
    def _per_person(g):
        g = _add_year_delta(g)
        g['_ANC_PREV'] = g[anc].shift(1)
        g['FLAG_ANC_BAISSE'] = ((g[anc] < g['_ANC_PREV']) & g[anc].notna() & g['_ANC_PREV'].notna()).astype('Int8')
        g['FLAG_ANC_VITESSE_IRR'] = (((g[anc] - g['_ANC_PREV']) > (g['_YEAR_DELTA'] + max_drift)) &
                                     g[anc].notna() & g['_ANC_PREV'].notna()).astype('Int8')
        return g.drop(columns=['_ANC_PREV'], errors='ignore')
    return df.groupby('MATRICULE', group_keys=False).apply(_per_person)




# CHARGEMENT DE LA BASE DE TRAVAIL

In [23]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"

# object_key 
object_key = "Solde/panel_solde_df_1_code.parquet"  # Chemin dans le bucket

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,SERVICE_NORM,CODE_SERVICE,ORGANISME_NORM,CODE_ORGANISME,CLASSE/ECHELON_NORM,CODE_CLASSE/ECHELON,EMPLOI_NORM,CODE_EMPLOI,SITUATION_NORM,CODE_SITUATION
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,<NA>,,<NA>,en activite,10
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,a affecter,1800,"min affaires etrangeres, de l integration afri...",25,,<NA>,,<NA>,en activite,10
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,dir personnel police nationale,3813,min de l interieur et de la securite,38,commissaire divis 1er ech,LW,commissaire de police,P1ZC,retraite pour limite age,60


In [37]:
df = panel_solde_df.copy()
# 1) Inspection rapide des types (optionnel)
print(df[["PERIODE", "DATE_COLLECTE"]].dtypes)

PERIODE                  object
DATE_COLLECTE    datetime64[ns]
dtype: object


In [16]:
import pandas as pd
import numpy as np

df = panel_solde_df.copy()

# 1) Parser PERIODE (int YYYYMM) -> Period[M]
p_str = pd.to_numeric(df["PERIODE"], errors="coerce").astype("Int64").astype(str).str.zfill(6)
year_p = pd.to_numeric(p_str.str[:4], errors="coerce")
month_p = pd.to_numeric(p_str.str[4:6], errors="coerce")
valid_p = month_p.between(1, 12)
periode_M = pd.Series(pd.NaT, index=df.index, dtype="period[M]")
periode_M[valid_p] = pd.PeriodIndex(year=year_p[valid_p], month=month_p[valid_p], freq="M")

# 2) Parser DATE_COLLECTE (str) -> datetime -> Period[M]
dc = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce") \
       .fillna(pd.to_datetime(df["DATE_COLLECTE"], errors="coerce", dayfirst=True))
date_M = dc.dt.to_period("M")

df["_PERIODE_M"] = periode_M
df["_DATE_COLLECTE_M"] = date_M

# 3) Comparaison au mois
same_month = df["_PERIODE_M"].eq(df["_DATE_COLLECTE_M"])
both_na = df["_PERIODE_M"].isna() & df["_DATE_COLLECTE_M"].isna()

print("=== Résumé ===")
print(f"Total: {len(df):,}")
print(f"Égalité mois: {same_month.sum():,} ({same_month.mean():.1%})")
print(f"Incohérences: {(~same_month & ~both_na).sum():,} ({(~same_month & ~both_na).mean():.1%})")

# 4) Écart en MOIS sans TypeError (méthode année*12 + mois)
valid = df["_PERIODE_M"].notna() & df["_DATE_COLLECTE_M"].notna()

month_diff = pd.Series(pd.NA, index=df.index, dtype="Int64")
y_p = df.loc[valid, "_PERIODE_M"].dt.year
m_p = df.loc[valid, "_PERIODE_M"].dt.month
y_d = df.loc[valid, "_DATE_COLLECTE_M"].dt.year
m_d = df.loc[valid, "_DATE_COLLECTE_M"].dt.month

month_diff.loc[valid] = (y_d*12 + m_d) - (y_p*12 + m_p)
df["_ECART_MOIS"] = month_diff

print("\n=== Distribution des écarts (mois) ===")
print(month_diff.value_counts(dropna=True).sort_index())

# 5) Écart en jours (DATE_COLLECTE - 1er jour de PERIODE), idem avec masque
ecart_jours = pd.Series(pd.NA, index=df.index, dtype="Int64")
periode_as_date = df["_PERIODE_M"].dt.to_timestamp(how="start")
ecart_jours.loc[valid] = (dc.loc[valid] - periode_as_date.loc[valid]).dt.days.astype("Int64")
df["_ECART_JOURS"] = ecart_jours

print("\n=== Écart en jours (stats) ===")
print(ecart_jours.dropna().describe())

# 6) Exemples d’incohérences
mismatch = df.loc[~same_month & ~both_na, ["PERIODE", "DATE_COLLECTE", "_PERIODE_M", "_DATE_COLLECTE_M", "_ECART_MOIS", "_ECART_JOURS"]]
print("\n=== Exemples d'incohérences (5) ===")
print(mismatch.head(5))


/tmp/ipykernel_134/58105453.py:12: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  periode_M[valid_p] = pd.PeriodIndex(year=year_p[valid_p], month=month_p[valid_p], freq="M")


=== Résumé ===
Total: 15,627,963
Égalité mois: 0 (0.0%)
Incohérences: 15,627,963 (100.0%)

=== Distribution des écarts (mois) ===
Series([], Name: count, dtype: Int64)

=== Écart en jours (stats) ===
count     0.0
mean     <NA>
std      <NA>
min      <NA>
25%      <NA>
50%      <NA>
75%      <NA>
max      <NA>
dtype: Float64

=== Exemples d'incohérences (5) ===
  PERIODE DATE_COLLECTE _PERIODE_M _DATE_COLLECTE_M  _ECART_MOIS  _ECART_JOURS
0  012015    2015-01-31        NaT          2015-01         <NA>          <NA>
1  012015    2015-01-31        NaT          2015-01         <NA>          <NA>
2  012015    2015-01-31        NaT          2015-01         <NA>          <NA>
3  012015    2015-01-31        NaT          2015-01         <NA>          <NA>
4  012015    2015-01-31        NaT          2015-01         <NA>          <NA>


In [16]:
print (panel_solde_df.columns)
print(f"Nous avons {len(panel_solde_df)} observations")

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION'],
      dtype='object')
Nous avons 15627963 observations


# APPEL DES DIFFERENTES FONCTIONS 

In [28]:
# 0) Construire DATE_REF si absente (utilise DATE_COLLECTE)
if 'DATE_REF' not in panel_solde_df.columns:
    panel_solde_df = _ensure_date_ref(panel_solde_df)

### Situation matrimoniale

In [29]:
panel_solde_df, rep_sitmat = build_situation_matrimoniale(panel_solde_df)
# Aperçu du tableau (effectifs + %)
rep_sitmat["table"]

,Effectif,Pourcentage (%)
SITUATION_MATRIMONIALE_RECODE,,
Autres,25466,0.16
Célibataire,10021194,64.12
Divorcé/Séparé,12637,0.08
Marié(e),5562568,35.59
Veuf/Veuve,6098,0.04


Colonnes présentes: {'SITUATION MATRIMONIALE'}


/tmp/ipykernel_1115/759897779.py:63: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('MATRICULE', group_keys=False).apply(_per_person)


#### Cohérence temporelle 

In [39]:
# ============================================================
# EXEMPLE D’UTILISATION
# ============================================================
# 1) Calculer les flags demandés sur ton DataFrame principal :
panel_solde_df = compute_selected_flags(panel_solde_df)

# 2) Exporter le rapport par catégorie uniquement pour ces flags :
export_selected_flags_report(
   panel_solde_df,
     outfile="rapport_flags_selection.xlsx",
     key_cols=("MATRICULE",),
     date_cols=("DATE_REF","DATE_COLLECTE"),
     extra_cols=("AGE","AGE_IMPUTE","AGE_VALIDE","SEXE","SEXE_IMPUTE","AGE_PRISERVICE","ANCIENNETE"),
     flag_map=FLAG_MAP
)

ValueError: Colonne MATRICULE manquante.

In [30]:
panel_solde_df = sitmat_coherence_temporelle(panel_solde_df)

In [36]:
# ======================= APPEL — SITUATION MATRIMONIALE =======================
sit_flag_cols = [
    'FLAG_SITMAT_INCONNU','FLAG_SITMAT_NON_MAPPEE',
    'FLAG_SITMAT_ENFANTS','FLAG_SITMAT_AGE',
    'FLAG_SITMAT_VARIATION','FLAG_SITMAT_VARIATIONS'
]
sit_extra_cols = [
    'SITUATION MATRIMONIALE','_SITMAT_NORM','_SITMAT_RECODE',
    'NB_ENFANTS','NOMBRE ENFANT','AGE','AGE_IMPUTE'
]

export_flags_report(
    panel_solde_df,
    flag_cols=sit_flag_cols,
    key_cols=['MATRICULE'],          # si absente, l’algo cherchera une clé équivalente; sinon ROW_ID
    extra_cols=sit_extra_cols,
    outfile='rapport_flags_sitmat',
    title='SITUATION MATRIMONIALE'
)

[INFO] Clé de groupement utilisée : ['MATRICULE||CODE_ORGANISME']

===== SITUATION MATRIMONIALE =====

Taux de flags (%) :
                  FLAG  TAUX_%
   FLAG_SITMAT_ENFANTS   36.97
 FLAG_SITMAT_VARIATION    0.37
FLAG_SITMAT_VARIATIONS    0.13
   FLAG_SITMAT_INCONNU     0.0
FLAG_SITMAT_NON_MAPPEE     0.0
       FLAG_SITMAT_AGE     0.0

Total lignes avec ≥1 flag : 5829860 / 15627963 (37.30%)

Top clés à problème :
MATRICULE||CODE_ORGANISME  NB_LIGNES_FLAGGEES
                611313X15                  72
                610670U15                  72
                611902X15                  72
                257885T22                  72
                257888E22                  72
                257892S22                  72
                257869J22                  72
                257868R22                  72
                257859Q22                  72
                257857E22                  72
                257856D22                  72
                290331C22   

ValueError: This sheet is too large! Your sheet size is: 5829860, 12 Max sheet size is: 1048576, 16384

In [ ]:
print (panel_solde_df.columns)

#### Cohérence non temporelle 

In [ ]:
panel_solde_df = sitmat_coherence_non_temporelle(panel_solde_df)

### Nombre d'enfants

In [17]:
panel_solde_df, rep_enfants = build_nombre_enfants(panel_solde_df)
# Aperçu du tableau (effectifs + %)
rep_enfants["table"]

,Effectif,Pourcentage (%)
NBRE_ENFTS_IMPUTE,,
0,5604678,35.86
1,2369618,15.16
2,2595972,16.61
3,2164280,13.85
4,1426419,9.13
5,780665,5.0
6,476700,3.05
7,133859,0.86
8,50576,0.32


In [ ]:
# =====================================================================================
# 2) ENFANTS — fonction puis appel
# =====================================================================================
def flags_enfants(df, age_col, max_enfants=12):
    df = df.copy()
    enfants_col = 'NOMBRE ENFANT' if 'NOMBRE ENFANT' in df.columns else ('NB_ENFANTS' if 'NB_ENFANTS' in df.columns else None)
    if enfants_col is None:
        df['NB_ENFANTS_TMP'] = np.nan
        enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    df['FLAG_ENFANTS_NEGATIF']  = (df[enfants_col] < 0).astype('Int8')
    df['FLAG_ENFANTS_OUTLIER']  = (df[enfants_col] > max_enfants).astype('Int8')
    df['FLAG_ENFANTS_MANQUANT'] = df[enfants_col].isna().astype('Int8')

    # Cohérences avec l'âge (si dispo)
    df['FLAG_ENFANTS_AGE']      = ((df[age_col] < 15) & (df[enfants_col] > 0)).astype('Int8')
    df['FLAG_ENFANTS_FERT_IRR'] = (df[enfants_col] > (df[age_col] - 12)).astype('Int8')
    return df, enfants_col

# Appel
panel_solde_df, _ENF_COL = flags_enfants(panel_solde_df, age_col=_AGE_COL, max_enfants=12)


#### Cohérence temporelle 

In [ ]:
panel_solde_df = enfants_coherence_temporelle(panel_solde_df)

#### Cohérence non temporelle 

In [ ]:
# 2) NOMBRE D’ENFANTS
panel_solde_df = enfants_coherence_non_temporelle(
    panel_solde_df,
    # age_col='AGE_IMPUTE',             # (optionnel) précise au besoin
    max_enfants=12,
    # enfants_impute_col='ENFANTS_IMPUTE'  # (optionnel) si tu as un flag d’imputation existant
)

# AFFICHAGE DES DIFFERENTES VARIABLES CREES 

## CATEGORIE 

In [16]:
# Appliquer la fonction de création des catégories 1 et 2
panel_solde_df = build_categories(panel_solde_df, grade_col="GRADE", grade2_col="GRADE 2")

# Aperçu
panel_solde_df[["GRADE","CATEGORIE_1","GRADE 2","CATEGORIE_2"]].head()

,GRADE,CATEGORIE_1,GRADE 2,CATEGORIE_2
0,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
1,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
2,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
3,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
4,Fonctionnaire Catégorie A7,A,A7,A


In [17]:
print(panel_solde_df['CODE_ORGANISME'].count())

15622889


### Extraction des modalités des variables GRADE et GRADE 2

In [17]:
# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE", "CATEGORIE_1"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE"] != "") & (unique_grades["CATEGORIE_1"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades.xlsx", index=False)


In [18]:
# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE 2", "CATEGORIE_2"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE 2"] != "") & (unique_grades["CATEGORIE_2"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades_2.xlsx", index=False)

In [19]:
# 📊 Créer un tableau croisé des effectifs
tableau_croise_effectifs = pd.crosstab(
    panel_solde_df["CATEGORIE_1"],   # Variable en ligne
    panel_solde_df["CATEGORIE_2"],   # Variable en colonne
    margins=True,        # Ajoute les totaux
    margins_name="Total" # Nom de la ligne/colonne total
)
tableau_croise_effectifs

CATEGORIE_2,Non Fonctionnaire,A,B,C,D,Total
CATEGORIE_1,,,,,,
Non Fonctionnaire,320356,43,0,0,0,320399
A,0,4797789,0,0,0,4797789
B,0,0,6119667,0,0,6119667
C,0,0,0,3961427,0,3961427
D,0,0,0,0,428681,428681
Total,320356,4797832,6119667,3961427,428681,15627963


In [20]:
print(((panel_solde_df["CATEGORIE_1"].isna()) | (panel_solde_df["CATEGORIE_1"].str.strip() == "")).sum())
print(((panel_solde_df["CATEGORIE_2"].isna()) | (panel_solde_df["CATEGORIE_2"].str.strip() == "")).sum())

0
0


In [21]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["EMPLOI_NORM"].value_counts()
print(tab_statut)

EMPLOI_NORM
                         36
medecin generaliste       5
inspecteur des impots     2
Name: count, dtype: int64


In [22]:
# Filtrer les lignes
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]

# Sélectionner les colonnes à exporter
colonnes_a_garder = ["MATRICULE", "PERIODE", "CATEGORIE_1", "CATEGORIE_2", "EMPLOI_NORM"]
export_df = filtre[colonnes_a_garder]

# 💾 Exporter vers Excel
nom_fichier = "filtre_non_fonctionnaire_grade_A.xlsx"
export_df.to_excel(nom_fichier, index=False)

print(f"✅ Fichier Excel créé : {nom_fichier}")


✅ Fichier Excel créé : filtre_non_fonctionnaire_grade_A.xlsx


### Cohérence non temporelle 

### Cohérence temporelle 

## STATUT

In [23]:
# Appliquer la fonction de création de la variable statut
panel_solde_df, rep = build_statut_from_cats(panel_solde_df)
rep

{'repartition_statut': STATUT
 Non Fonctionnaire      315125
 Fonctionnaire        15307347
 Cas particulier          5491
 Name: count, dtype: int64}

In [24]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["STATUT"].value_counts()
print(tab_statut)

STATUT
Cas particulier      43
Non Fonctionnaire     0
Fonctionnaire         0
Name: count, dtype: int64


In [25]:
# Tabulation simple sur STATUT
tab_statut = panel_solde_df["STATUT"].value_counts(dropna=False).sort_index()

# Total lignes
total_lignes = tab_statut.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut": tab_statut.index,
    "Effectif": tab_statut.values,
    "Proportion (%)": (tab_statut.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut,Effectif,Proportion (%)
0,Non Fonctionnaire,315125,2.02
1,Fonctionnaire,15307347,97.95
2,Cas particulier,5491,0.04
3,Total,15627963,100.01


## STATUT DEFINITIF

In [26]:
# Appliquer la fonction de création du statut définitif en s'appuyant sur la période MMYYY
panel_solde_df, rep = build_statut_def_periode(panel_solde_df, return_report=True)
rep

{'Statut définifitif': Statut_def_mois
 Non Fonctionnaire      231219
 Cas particulier          4964
 Fonctionnaire        15391780
 Name: count, dtype: int64}

In [27]:
# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut définifitif"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut Définitif,Effectif,Proportion (%)
0,Non Fonctionnaire,231219,1.48
1,Cas particulier,4964,0.03
2,Fonctionnaire,15391780,98.49
3,Total,15627963,100.00


In [28]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["Statut_def_mois"].value_counts()
print(tab_statut)

Statut_def_mois
Cas particulier      43
Non Fonctionnaire     0
Fonctionnaire         0
Name: count, dtype: int64


In [29]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Vérifier si un même matricule apparaît plusieurs fois dans la même période
doublons = cas_particuliers_df[cas_particuliers_df.duplicated(subset=["MATRICULE", "PERIODE"], keep=False)]

if doublons.empty:
    print("✅ Chaque matricule est unique par période.")
else:
    print(f"⚠️ {len(doublons)} doublons trouvés (mêmes matricules dans la même période).")
    print(doublons[["MATRICULE", "PERIODE"]].sort_values(by=["MATRICULE", "PERIODE"]))

⚠️ 2 doublons trouvés (mêmes matricules dans la même période).
        MATRICULE PERIODE
9228788   089081R  102018
9228789   089081R  102018


In [30]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_mois.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1" , "GRADE", "CATEGORIE_2", "GRADE 2", "PERIODE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")

Fichier exporté : cas_particuliers_def_mois.xlsx avec 4964 observations


#### Point de contôle

In [ ]:

# =====================================================================================
# 5) STATUT & STATUT_DÉFINITIF — fonction puis appel
# =====================================================================================
def flags_statuts(df, age_col, statut_ok=None, map_statut_to_def=None, retraite_min_age=50):
    df = df.copy()
    statut_ok = set(statut_ok or {
        'en activite','disponibilite','detachement','suspension','fin de contrat',
        "fin d'engagement",'retraite','retraite pour limite age','retraite pour anciennete',
        'deces','arret recensement 2011','abandon de poste','regul. indemnites',
        'sous controle // traitement','arret police','viol'
    })
    map_statut_to_def = map_statut_to_def or {
        'en activite':'fonctionnaire','detachement':'fonctionnaire','disponibilite':'fonctionnaire',
        'suspension':'fonctionnaire','retraite':'fonctionnaire','retraite pour limite age':'fonctionnaire',
        'retraite pour anciennete':'fonctionnaire','fin de contrat':'non-fonctionnaire'
    }

    s_col = 'STATUT' if 'STATUT' in df.columns else None
    if s_col is None:
        df['STATUT_TMP'] = np.nan; s_col = 'STATUT_TMP'
    sd_col = 'STATUT_DEFINITIF' if 'STATUT_DEFINITIF' in df.columns else None
    if sd_col is None:
        df['STATUT_DEFINITIF_TMP'] = np.nan; sd_col = 'STATUT_DEFINITIF_TMP'

    df['_STATUT_NORM']     = df[s_col].map(_norm_txt)
    df['_STATUT_DEF_NORM'] = df[sd_col].map(_norm_txt)

    df['FLAG_STATUT_NON_MAPPE'] = (~df['_STATUT_NORM'].isin(statut_ok) & df['_STATUT_NORM'].notna()).astype('Int8')

    expected_def = df['_STATUT_NORM'].map(map_statut_to_def).fillna('cas particulier')
    df['FLAG_STATUT_DEF_INCOHERENT'] = ((df['_STATUT_DEF_NORM'].notna()) & (df['_STATUT_DEF_NORM'] != expected_def)).astype('Int8')
    df['FLAG_STATUT_DEF_INCONNU']    = df['_STATUT_DEF_NORM'].isna().astype('Int8')

    _stat_norm_str = df['_STATUT_NORM'].astype('string')
    df['FLAG_AGE_RETRAITE_IRR'] = (_stat_norm_str.str.contains('retraite', na=False) & (df[age_col] < retraite_min_age)).astype('Int8')
    return df

# Appel
panel_solde_df = flags_statuts(panel_solde_df, age_col=_AGE_COL, retraite_min_age=50)

### Cohérence temporelle 

In [ ]:
panel_solde_df = statut_coherence_temporelle(panel_solde_df)


### Cohérence non temporelle 

In [ ]:
# --- Appel (STATUT)
panel_solde_df = statut_coherence_non_temporelle(panel_solde_df)

In [31]:
df = panel_solde_df.copy()

# 0) Dates et année
df["DATE_COLLECTE"] = pd.to_datetime(df["DATE_COLLECTE"], errors="coerce", dayfirst=True)
if "ANNEE_COLLECTE" not in df.columns:
    df["ANNEE_COLLECTE"] = df["DATE_COLLECTE"].dt.year

# 1) Créer Statut_def_annee si absent (mode de Statut_def_mois par MATRICULE × ANNEE_COLLECTE)
if "Statut_def_annee" not in df.columns:
    if "Statut_def_mois" not in df.columns:
        raise KeyError("Ni 'Statut_def_annee' ni 'Statut_def_mois' présents dans le DataFrame.")

    mode_par_annee = (
        df.groupby(["MATRICULE", "ANNEE_COLLECTE"])["Statut_def_mois"]
          .agg(lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA)
          .rename("Statut_def_annee")
          .reset_index()
    )
    df = df.merge(mode_par_annee, on=["MATRICULE", "ANNEE_COLLECTE"], how="left")

# 2) Fonction utilitaire d’export de conflits
def export_conflits(df_in, statut_col, suffix):
    tmp = (
        df_in.groupby("MATRICULE")[statut_col]
             .nunique(dropna=True)
             .reset_index(name="nb_statuts_distincts")
    )
    problemes = tmp[tmp["nb_statuts_distincts"] > 1]
    nb = problemes.shape[0]
    print(f"Nombre de matricules ayant plusieurs statuts ({suffix}) : {nb}")

    if nb > 0:
        details = df_in[df_in["MATRICULE"].isin(problemes["MATRICULE"])]
        cols = ["MATRICULE", "DATE_COLLECTE", statut_col]
        export_df = details[cols].sort_values(["MATRICULE", "DATE_COLLECTE"])
        out = f"matricules_statut_problemes_{suffix}.xlsx"
        export_df.to_excel(out, index=False)
        print(f"✅ Export terminé : {out}")
    else:
        print("✅ Aucun problème détecté (un seul statut par matricule).")
    return nb

# 3) Exécutions
export_conflits(df, "Statut_def_mois", "mois")
export_conflits(df, "Statut_def_annee", "annee")


Nombre de matricules ayant plusieurs statuts (mois) : 896
✅ Export terminé : matricules_statut_problemes_mois.xlsx
Nombre de matricules ayant plusieurs statuts (annee) : 711
✅ Export terminé : matricules_statut_problemes_annee.xlsx


711

In [32]:
# Appliquer la fonction de création du statut définitif en s'appuyant sur l'année
panel_solde_df, rep = build_statut_def_annee(panel_solde_df, return_report=True)
rep

{'Statut_def_distribution': Statut_def_annee
 Non Fonctionnaire      230930
 Cas particulier          3469
 Fonctionnaire        15393564
 Name: count, dtype: int64}

In [33]:
# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut_def_distribution"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut Définitif,Effectif,Proportion (%)
0,Non Fonctionnaire,230930,1.48
1,Cas particulier,3469,0.02
2,Fonctionnaire,15393564,98.50
3,Total,15627963,100.00


In [34]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A") & 
    (panel_solde_df["EMPLOI_NORM"] != "") 

]
tab_statut = filtre["Statut_def_annee"].value_counts()
print(tab_statut)

Statut_def_annee
Fonctionnaire        7
Non Fonctionnaire    0
Cas particulier      0
Name: count, dtype: int64


In [35]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_annee"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_annee.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1", "CATEGORIE_2", "ANNEE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")

Fichier exporté : cas_particuliers_def_annee.xlsx avec 3469 observations


## SEXE

In [36]:
# 1) Construire la variable propre
panel_solde_df = build_sexe_clean(
    panel_solde_df,
    id_col="MATRICULE",
    sex_col="SEXE",
    collect_col="DATE_COLLECTE",
    new_col="SEXE_CLEAN"
)

# 2) Imputer à partir de SEXE_CLEAN (seul changement: sex_col="SEXE_CLEAN")
panel_solde_df, report = imputer_sexe(
    panel_solde_df,
    sex_col="SEXE_CLEAN",         # ← avant: "SEXE"
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_IMPUTE",
    debug=True
)

report


=== Statistiques avant imputation (SEXE_CLEAN) ===


,Effectif
SEXE_CLEAN,
Féminin,4684666
Masculin,10943276
NaN,21



=== Statistiques après imputation (SEXE_IMPUTE) ===


,Effectif
SEXE_IMPUTE,
Féminin,4684666
Masculin,10943297



=== Pourcentages après imputation ===


,Pourcentage (%)
SEXE_IMPUTE,
Féminin,29.976178
Masculin,70.023822



=== Tableau croisé avant/après ===


SEXE_IMPUTE,Féminin,Masculin,Total
SEXE_CLEAN,,,
Féminin,4684666,0,4684666.0
Masculin,0,10943276,10943276.0
NaN,0,21,NaN
Total,4684666,10943297,15627963.0


{'tab_sexe_avant': SEXE_CLEAN
 Féminin      4684666
 Masculin    10943276
 NaN               21
 Name: count, dtype: int64,
 'tab_sexe_apres': SEXE_IMPUTE
 Féminin      4684666
 Masculin    10943297
 Name: count, dtype: int64,
 'tab_sexe_apres_pct': SEXE_IMPUTE
 Féminin     29.976178
 Masculin    70.023822
 Name: proportion, dtype: float64,
 'crosstab_sexe': SEXE_IMPUTE  Féminin  Masculin       Total
 SEXE_CLEAN                                
 Féminin      4684666         0   4684666.0
 Masculin           0  10943276  10943276.0
 NaN                0        21         NaN
 Total        4684666  10943297  15627963.0}

In [ ]:
# =====================================================================================
# 4) SEXE — fonction puis appel
# =====================================================================================
def flags_sexe(df):
    df = df.copy()
    src = 'SEXE' if 'SEXE' in df.columns else None
    if src is None:
        df['SEXE_TMP'] = np.nan
        src = 'SEXE_TMP'
    df['_SEXE_NORM'] = _sex_norm(df[src])
    df['FLAG_SEXE_INVALIDE'] = (~df['_SEXE_NORM'].isin(['M','F'])).fillna(True).astype('Int8')
    df['FLAG_SEXE_INCONNU']  = df['_SEXE_NORM'].isna().astype('Int8')
    return df

# Appel
panel_solde_df = flags_sexe(panel_solde_df)

### Cohérence non temporelle 

In [ ]:
panel_solde_df = sexe_coherence_non_temporelle(panel_solde_df)

### Cohérence temporelle 

In [ ]:
panel_solde_df = sexe_coherence_temporelle(panel_solde_df)

#### Phase contrôle 

##### Verifier que le sexe unique 

In [37]:
# Regrouper par matricule et compter le nombre de sexes distincts
sexe_par_matricule = (
    panel_solde_df.groupby("MATRICULE")["SEXE_CLEAN"]
    .nunique()
    .reset_index(name="nb_sexes_distincts")
)

# Filtrer ceux qui ont plus d’un sexe (incohérents)
problemes_sexe = sexe_par_matricule[sexe_par_matricule["nb_sexes_distincts"] > 1]

# Afficher les matricules problématiques
print("Matricules ayant plusieurs sexes enregistrés :")
print(problemes_sexe)


Matricules ayant plusieurs sexes enregistrés :
Empty DataFrame
Columns: [MATRICULE, nb_sexes_distincts]
Index: []


## AGE 

### AGE Valide

In [38]:
# 0) S'assurer qu'on a SEXE_IMPUTE
if "SEXE_IMPUTE" not in panel_solde_df.columns:
    panel_solde_df, _ = imputer_sexe(
        panel_solde_df,
        sex_col="SEXE",
        collect_col="DATE_COLLECTE",
        sex_valid_col="SEXE_IMPUTE",
        debug=False
    )

# 1) Calcul des âges (build_age_valide ne renvoie que le df)
panel_solde_df = build_age_valide(
    panel_solde_df,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    age_min=18, age_max=70,
    do_impute_age=True,
    stabilize_age=True,           # ← active AGE_IMPUTE_STABLE
    stabilize_method="last"       # 'last' recommandé; sinon 'mode' ou 'median'
)

# 2) Reconstruire les tableaux "rep" à partir du df
tab_sexe_avant      = panel_solde_df["SEXE"].value_counts(dropna=False).sort_index()
tab_sexe_avant_pct  = panel_solde_df["SEXE"].value_counts(normalize=True, dropna=False).sort_index() * 100
tab_sexe_apres      = panel_solde_df["SEXE_IMPUTE"].value_counts(dropna=False).sort_index()
tab_sexe_apres_pct  = panel_solde_df["SEXE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index() * 100
crosstab_sexe       = pd.crosstab(panel_solde_df["SEXE"], panel_solde_df["SEXE_IMPUTE"], dropna=False, margins=True, margins_name="Total")

rep = {
    "tab_sexe_avant": tab_sexe_avant,
    "tab_sexe_avant_pct": tab_sexe_avant_pct,
    "tab_sexe_apres": tab_sexe_apres,
    "tab_sexe_apres_pct": tab_sexe_apres_pct,
    "crosstab_sexe": crosstab_sexe
}

# 3) Afficher les tableaux
print(pd.DataFrame({"Effectif": rep["tab_sexe_avant"], "Pourcentage (%)": rep["tab_sexe_avant_pct"]}))
print(pd.DataFrame({"Effectif": rep["tab_sexe_apres"],  "Pourcentage (%)": rep["tab_sexe_apres_pct"]}))
print(rep["crosstab_sexe"])

# 4) Aperçu des colonnes ajoutées
cols_preview = [
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
    "ANNEE_COLLECTE","MOIS_COLLECTE",
    "SEXE","SEXE_IMPUTE",
    "AGE","AGE_VALIDE","AGE_IMPUTE"
]
print(panel_solde_df[cols_preview].head())


[AGE] Matricules avec AGE_IMPUTE variable : 262,106 (95.29%)
[AGE] Stabilisation : last
          Effectif  Pourcentage (%)
SEXE                               
Féminin    4683169        29.966599
Masculin  10944644        70.032441
None           150         0.000960
             Effectif  Pourcentage (%)
SEXE_IMPUTE                           
Féminin       4684666        29.976178
Masculin     10943297        70.023822
SEXE_IMPUTE  Féminin  Masculin       Total
SEXE                                      
Féminin      4678407      4762   4683169.0
Masculin        6133  10938511  10944644.0
NaN              126        24         NaN
Total        4684666  10943297  15627963.0
  MATRICULE DATE_NAISSANCE_CORRIGEE  ANNEE_COLLECTE  MOIS_COLLECTE      SEXE  \
0   011242X              1939-01-01            2015              1  Masculin   
1   012856Q              1939-01-01            2015              1  Masculin   
2   013454N              1924-01-01            2015              1  Masculin  

In [39]:
# Étape 1 : Supprimer les doublons sur la colonne AGE
filtered_df = panel_solde_df.drop_duplicates(subset=["AGE"])

# Étape 2 : Définir le chemin et le nom du fichier Excel de sortie
output_file = "Age.xlsx"

# Étape 3 : Exporter uniquement les colonnes pertinentes
filtered_df[["AGE", "AGE_VALIDE", "AGE_IMPUTE"]].to_excel(output_file, index=False)

# Étape 4 : Confirmation
print(f"Fichier exporté : {output_file}")


Fichier exporté : Age.xlsx


In [40]:
# Compter les effectifs d'âge (en incluant les NaN)
tab_age = panel_solde_df["AGE_IMPUTE"].value_counts(dropna=False).sort_index()

# Calculer les proportions (%)
tab_age_pct = (tab_age / tab_age.sum()) * 100

# Créer le tableau final
tab_age_final = pd.DataFrame({
    "Effectif": tab_age,
    "Proportion (%)": tab_age_pct.round(2)
})

# Afficher tout le tableau, y compris les NaN
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(tab_age_final)

            Effectif  Proportion (%)
AGE_IMPUTE                          
18                62             0.0
19               289             0.0
20              1196            0.01
21              3443            0.02
22              7914            0.05
23             15461             0.1
24             29726            0.19
25             52948            0.34
26             88111            0.56
27            135793            0.87
28            195264            1.25
29            266215             1.7
30            342633            2.19
31            422902            2.71
32            499145            3.19
33            574993            3.68
34            633630            4.05
35            683297            4.37
36            723463            4.63
37            739369            4.73
38            743748            4.76
39            737842            4.72
40            718132             4.6
41            681719            4.36
42            633057            4.05
4

In [41]:
print(panel_solde_df["AGE_IMPUTE"].isna().sum())

0


In [ ]:
# =====================================================================================
# 1) ÂGE — fonction puis appel
# =====================================================================================
def flags_age(df, age_bounds=(18,75)):
    df = df.copy()
    age_col = 'AGE_IMPUTE' if 'AGE_IMPUTE' in df.columns else ('AGE' if 'AGE' in df.columns else None)
    if age_col is None:
        df['AGE_TMP'] = np.nan
        age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
    amin, amax = age_bounds
    df['FLAG_AGE_OUTLIER'] = ((df[age_col] < amin) | (df[age_col] > amax)).astype('Int8')
    return df, age_col

# Appel
panel_solde_df, _AGE_COL = flags_age(panel_solde_df, age_bounds=(18,75))


### Cohérence non temporelle 

In [ ]:
panel_solde_df = age_coherence_non_temporelle(panel_solde_df, bounds=CFG['age_bounds'])


### Cohérence temporelle 

In [ ]:
panel_solde_df = age_coherence_temporelle(panel_solde_df, max_drift=CFG['max_age_drift_years'])

## AGE DE PRISE DE SERVICE 

In [ ]:
# =====================================================================================
# 6) ÂGE DE PRISE DE SERVICE — fonction puis appel
# =====================================================================================
def flags_age_prise_service(df, age_col, age_ps_bounds=(14,60)):
    df = df.copy()
    ps_col = 'AGE_PRISERVICE' if 'AGE_PRISERVICE' in df.columns else None
    if ps_col is None:
        if {'DATE_PRISERVICE','DATE_NAISSANCE'}.issubset(df.columns):
            dps = pd.to_datetime(df['DATE_PRISERVICE'], errors='coerce')
            dob = pd.to_datetime(df['DATE_NAISSANCE'], errors='coerce')
            df['AGE_PS_CALC'] = ((dps - dob).dt.days / 365.25)
            ps_col = 'AGE_PS_CALC'
        else:
            df['AGE_PS_TMP'] = np.nan; ps_col = 'AGE_PS_TMP'
    df[ps_col] = pd.to_numeric(df[ps_col], errors='coerce')

    pmin, pmax = age_ps_bounds
    df['FLAG_AGE_PS_OUTLIER'] = ((df[ps_col] < pmin) | (df[ps_col] > pmax)).astype('Int8')
    df['FLAG_AGE_PS_GE_AGE']  = ((df[ps_col] >= df[age_col]) & df[ps_col].notna() & df[age_col].notna()).astype('Int8')
    return df, ps_col

# Appel
panel_solde_df, _AGE_PS_COL = flags_age_prise_service(panel_solde_df, age_col=_AGE_COL, age_ps_bounds=(14,60))

# ==

### Cohérence temporelle 

In [44]:
panel_solde_df, rep_srv = build_age_service(
    panel_solde_df,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    age_min=18, age_max=40,
    days_per_year=365,
    age_src_priority=("AGE_IMPUTE_STABLE","AGE_IMPUTE","AGE"),
    stabilize_service=True,
    stabilize_method="mode",      # 'mode' robuste pour lisser les petits écarts
    return_tables=True,
    sample_anomalies=10,
    recompute_corrected=True
)

print("→ Source d'âge utilisée pour le service :", rep_srv.get("age_source_used"))

[SERVICE] Multi-âges imputés AVANT stabilisation : 261,829 / 275,066 (95.19%)
[SERVICE] Multi-âges APRÈS  stabilisation : 0 / 275,066 (0.0%)  → méthode=mode
→ Source d'âge utilisée pour le service : AGE_IMPUTE_STABLE


### Cohérence non temmporelle

In [ ]:
panel_solde_df = ageps_coherence_non_temporelle(panel_solde_df, bounds=CFG['age_ps_bounds'])


### Cohérence temmporelle 

In [ ]:
panel_solde_df = ageps_coherence_temporelle(panel_solde_df)

In [45]:
# c) CONTRÔLE : vérifier que l’âge au service est constant par matricule
#    (AGE_AU_SERVICE_FINAL doit avoir 1 seule valeur par matricule)

nb_multi = (
    panel_solde_df.groupby("MATRICULE")["AGE_AU_SERVICE_FINAL"]
                  .nunique(dropna=True).gt(1).sum()
)
tot_mats = panel_solde_df["MATRICULE"].nunique()
pct = round(nb_multi / tot_mats * 100, 2)

if nb_multi == 0:
    print("✅ L'âge au service est constant par matricule (0 matricule multi-valeurs).")
else:
    print(f"⚠️  {nb_multi:,} matricules ({pct}%) ont plusieurs âges au service FINAL.")
    # échantillon pour audit rapide :
    ex_mats = (
        panel_solde_df.groupby("MATRICULE")["AGE_AU_SERVICE_FINAL"]
                      .nunique(dropna=True).reset_index(name="n")
                      .query("n>1").head(10)["MATRICULE"]
                      .tolist()
    )
    print("Exemples:", ex_mats)


✅ L'âge au service est constant par matricule (0 matricule multi-valeurs).


In [46]:
# 1) Âge source varie ?
diag_age_src = (
    panel_solde_df.groupby("MATRICULE")["AGE_IMPUTE"]
    .nunique(dropna=True).reset_index(name="n_age_stable")
)
print("Matricules sans AGE_IMPUTE (ou non unique) :",
      (diag_age_src["n_age_stable"] != 1).sum())

# 2) PDS multiples ?
pds_multi = (
    panel_solde_df.groupby("MATRICULE")["PRISE_DE_SERVICE_CORRIGEE"]
    .nunique(dropna=True).gt(1).sum()
)
print("Matricules avec plusieurs PDS corrigées :", pds_multi)

# 3) Part de lignes imputées et nb de groupes d'imputation traversés
tmp = panel_solde_df.copy()
tmp["is_imputed_srv"] = tmp["AGE_AU_SERVICE_VALIDE"].isna().astype(int)
tmp["grp_imp"] = list(zip(tmp["ANNEE_COLLECTE"], tmp["MOIS_COLLECTE"], tmp["SEXE_IMPUTE"]))

agg = (tmp.groupby("MATRICULE")
         .agg(pct_impute=("is_imputed_srv", "mean"),
              nb_grp_imp=("grp_imp", lambda x: len(set([g for g in x if pd.notna(g[2])])))))
print(agg.describe().round(3).T)


Matricules sans AGE_IMPUTE (ou non unique) : 262106
Matricules avec plusieurs PDS corrigées : 0
               count    mean     std  min   25%   50%   75%   max
pct_impute  275066.0   0.056   0.198  0.0   0.0   0.0   0.0   1.0
nb_grp_imp  275066.0  56.503  22.942  1.0  42.0  72.0  72.0  72.0


In [47]:
# 1) Âge source varie ?
diag_age_src = (
    panel_solde_df.groupby("MATRICULE")["DATE_COLLECTE"]
    .nunique(dropna=True).reset_index(name="n_age_stable")
)
print("date de collecte (ou non unique) :",
      (diag_age_src["n_age_stable"] != 1).sum())

date de collecte (ou non unique) : 274999


In [48]:
# 1) AGE_IMPUTE varie ?
nb_ageimp_var = (
    panel_solde_df.groupby("MATRICULE")["AGE_IMPUTE"]
    .nunique(dropna=True).gt(1).sum()
)
print("Matricules avec AGE_IMPUTE variable :", nb_ageimp_var)

# 2) PDS corrigée unique ?
nb_pds_multi = (
    panel_solde_df.groupby("MATRICULE")["PRISE_DE_SERVICE_CORRIGEE"]
    .nunique(dropna=True).gt(1).sum()
)
print("Matricules avec plusieurs PDS corrigées :", nb_pds_multi)

# 3) Contrôle final attendu : AGE_AU_SERVICE_FINAL doit être constant
nb_multi_final = (
    panel_solde_df.groupby("MATRICULE")["AGE_AU_SERVICE_FINAL"]
    .nunique(dropna=True).gt(1).sum()
)
tot = panel_solde_df["MATRICULE"].nunique()
print(f"Matricules encore multi-valeurs (FINAL) : {nb_multi_final} / {tot}")


Matricules avec AGE_IMPUTE variable : 262106
Matricules avec plusieurs PDS corrigées : 0
Matricules encore multi-valeurs (FINAL) : 0 / 275066


In [69]:
# Étape 1 : compter le nombre d'âges au service imputés distincts par matricule
multi_age = (
    panel_solde_df.groupby("MATRICULE")["AGE_AU_SERVICE_FINAL"]
    .nunique(dropna=True)
    .reset_index(name="nb_age_service_impute")
)

# Étape 2 : filtrer ceux qui en ont plusieurs
multi_age = multi_age[multi_age["nb_age_service_impute"] > 1]

# Étape 3 : voir les lignes correspondantes
details_multi_age = (
    panel_solde_df[panel_solde_df["MATRICULE"].isin(multi_age["MATRICULE"])]
    .sort_values(["MATRICULE", "DATE_COLLECTE"])
    [["MATRICULE", "DATE_COLLECTE", "AGE_AU_SERVICE_FINAL",
      "PRISE_DE_SERVICE_CORRIGEE", "FLAG_MULTI_AGE_SERVICE"]]
)

# Étape 4 : bilan global
nb_multi = len(multi_age)
nb_total = panel_solde_df["MATRICULE"].nunique()
pct_multi = round((nb_multi / nb_total) * 100, 2)

if nb_multi > 0:
    print(f"⚠️  {nb_multi} matricules ({pct_multi}%) présentent plusieurs âges au service imputés.")
    print("Voici un aperçu des premiers cas problématiques :")
    display(details_multi_age.head(10))
else:
    print(f"✅ Aucun matricule ne présente de variation d’âge au service imputé (sur {nb_total} individus).")


⚠️  261829 matricules (95.19%) présentent plusieurs âges au service imputés.
Voici un aperçu des premiers cas problématiques :


,MATRICULE,DATE_COLLECTE,AGE_AU_SERVICE_FINAL,PRISE_DE_SERVICE_CORRIGEE,FLAG_MULTI_AGE_SERVICE
0,011242X,2015-01-31,33.0,1969-01-01,1
184890,011242X,2015-02-28,33.0,1969-01-01,1
370226,011242X,2015-03-31,33.0,1969-01-01,1
556520,011242X,2015-04-30,33.0,1969-01-01,1
744494,011242X,2015-05-31,33.0,1969-01-01,1
934474,011242X,2015-06-30,33.0,1969-01-01,1
1126818,011242X,2015-07-31,33.0,1969-01-01,1
1319932,011242X,2015-08-31,32.0,1969-01-01,1
1514281,011242X,2015-09-30,32.0,1969-01-01,1
1709249,011242X,2015-10-31,32.0,1969-01-01,1


## ANCIENNETE

#### Ancienneté dans la fonction

In [67]:
panel_solde_df_mois, rep_mois = build_anciennete_mensuelle(
    panel_solde_df,
    matricule_col="MATRICULE",
    org_col="CODE_ORGANISME",
    collect_col="DATE_COLLECTE",
    start_service_col="PRISE_DE_SERVICE_CORRIGEE",
    min_period="2015-01",
    max_period="2020-12",
    return_by_org=True,
    return_by_person=True,
    keep_counts=True,
    include_prewindow_base=True
)


/tmp/ipykernel_131/1626648108.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["BASE_PRE2015_MOIS_PERS"] = out["BASE_PRE2015_MOIS_PERS"].fillna(0).astype(int)


In [69]:
cols_priorite = [
    "MATRICULE","ID_ORGANISME","PERIODE",
    "NB_APPA_MOIS_ORG","PRESENCE_MOIS_ORG","ANCIENNETE_MOIS_ORG","ANCIENNETE_AN_ORG",
    "NB_APPA_MOIS_PERS","PRESENCE_MOIS_PERS","ANCIENNETE_MOIS_PERS",
    "BASE_PRE2015_MOIS_PERS","ANCIENNETE_MOIS_PERS_TOTAL","ANCIENNETE_AN_PERS_TOTAL",
    "PRISE_DE_SERVICE_FIXE","START_AVANT_FENETRE"
]
cols_show = [c for c in cols_priorite if c in panel_solde_df_mois.columns]
print("\nAperçu colonnes clés :")
display(panel_solde_df_mois[cols_show].sort_values([c for c in ["MATRICULE","PERIODE"] if c in cols_show]).head(30))


Aperçu colonnes clés :


,MATRICULE,PERIODE,NB_APPA_MOIS_ORG,PRESENCE_MOIS_ORG,ANCIENNETE_MOIS_ORG,ANCIENNETE_AN_ORG,NB_APPA_MOIS_PERS,PRESENCE_MOIS_PERS,ANCIENNETE_MOIS_PERS,BASE_PRE2015_MOIS_PERS,ANCIENNETE_MOIS_PERS_TOTAL,ANCIENNETE_AN_PERS_TOTAL,PRISE_DE_SERVICE_FIXE,START_AVANT_FENETRE
0,011242X,2015-01-01,1,1,1,0,1,1,1,0,1,0,NaT,False
184890,011242X,2015-02-01,1,1,2,0,1,1,2,0,2,0,NaT,False
370226,011242X,2015-03-01,1,1,3,0,1,1,3,0,3,0,NaT,False
556520,011242X,2015-04-01,1,1,4,0,1,1,4,0,4,0,NaT,False
744494,011242X,2015-05-01,1,1,5,0,1,1,5,0,5,0,NaT,False
934474,011242X,2015-06-01,1,1,6,0,1,1,6,0,6,0,NaT,False
1126818,011242X,2015-07-01,1,1,7,0,1,1,7,0,7,0,NaT,False
1319932,011242X,2015-08-01,1,1,8,0,1,1,8,0,8,0,NaT,False
1514281,011242X,2015-09-01,1,1,9,0,1,1,9,0,9,0,NaT,False
1709249,011242X,2015-10-01,1,1,10,0,1,1,10,0,10,0,NaT,False


In [74]:
tab_start = panel_solde_df["START_AVANT_FENETRE"].value_counts(dropna=False)
print(tab_start)

START_AVANT_FENETRE
False    15627963
Name: count, dtype: int64


In [77]:
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION',
       'CATEGORIE_1', 'CATEGORIE_2', 'STATUT', 'Statut_def_mois', 'ANNEE',
       'Statut_def_annee', 'ANNEE_COLLECTE', 'MOIS_COLLECTE', 'SEXE_IMPUTE',
       'SEXE_CLEAN', 'DATE_NAISSANCE_CLEAN', 'DATE_NAISSANCE_CORRIGEE', 'AGE',
       'AGE_VALIDE', 'AGE_IMPUTE', 'AGE_IMPUTE_STAB

In [ ]:

# ==================================================================================
# 7) ANCIENNETÉ — fonction puis appel
# =====================================================================================
def flags_anciennete(df):
    df = df.copy()
    anc_col = 'ANCIENNETE' if 'ANCIENNETE' in df.columns else None
    if anc_col is None:
        df['ANCIENNETE_TMP'] = np.nan; anc_col = 'ANCIENNETE_TMP'
    df[anc_col] = pd.to_numeric(df[anc_col], errors='coerce')
    df['FLAG_ANC_NEGATIVE'] = (df[anc_col] < 0).astype('Int8')
    return df, anc_col

# Appel
panel_solde_df, _ANC_COL = flags_anciennete(panel_solde_df)


### Cohérence temporelle 

In [ ]:
panel_solde_df = anc_coherence_temporelle(panel_solde_df, max_drift=CFG['max_anc_drift_years'])

### Cohérence non temporelle 

In [ ]:
# --- Appel (ANCIENNETÉ)
panel_solde_df = anc_coherence_non_temporelle(panel_solde_df)

#### Vérification de la croissance de l'anciennété 

In [71]:
import pandas as pd

dfm = panel_solde_df_mois.copy()

# --- Colonne d'ancienneté à utiliser (priorité TOTAL, sinon mensuelle simple)
sen_col = "ANCIENNETE_MOIS_PERS_TOTAL" if "ANCIENNETE_MOIS_PERS_TOTAL" in dfm.columns else "ANCIENNETE_MOIS_PERS"
if sen_col not in dfm.columns:
    raise KeyError("Aucune colonne d'ancienneté trouvée (ANCIENNETE_MOIS_PERS_TOTAL ni ANCIENNETE_MOIS_PERS).")

# --- Année à partir de PERIODE
if not pd.api.types.is_period_dtype(dfm["PERIODE"]):
    dfm["PERIODE"] = pd.PeriodIndex(dfm["PERIODE"].astype(str), freq="M")
dfm["ANNEE"] = dfm["PERIODE"].dt.year

# --- Ancienneté de fin d'année par (matricule, année)
year_end = (
    dfm.groupby(["MATRICULE","ANNEE"], as_index=False)[sen_col]
       .max()  # fin d'année = max mensuel
       .sort_values(["MATRICULE","ANNEE"])
)

# --- Deltas année → année par matricule
year_end["DELTA"] = year_end.groupby("MATRICULE")[sen_col].diff()

# Règles de décision (globales) :
# 1) s'il existe au moins une baisse (DELTA < 0) → "décroît"
any_negative = (year_end["DELTA"] < 0).any()

# 2) sinon, si pour chaque matricule tous les deltas non-nuls sont > 0 → "évolue"
#    (les matricules avec un seul millésime n'ont pas de delta → considérés OK)
evolue_tous = (
    year_end.groupby("MATRICULE")["DELTA"]
            .apply(lambda s: s.dropna().gt(0).all())
            .all()
)

# 3) sinon, si pour chaque matricule tous les deltas sont == 0 → "constant"
constant_tous = (
    year_end["DELTA"].fillna(0).eq(0).all()
)

# --- Affichage du message unique
if any_negative:
    print("L'ancienneté decroit.")
elif evolue_tous:
    print("L'ancienneté évolue chaque année pour chaque matricule.")
elif constant_tous:
    print("L'ancienneté est constant chaque année pour chaque matricule.")
else:
    # Cas mixte (ni toute en hausse, ni toute constante, mais sans baisse explicite)
    # Par consigne, on bascule sur "décroît".
    print("L'ancienneté decroit.")


/tmp/ipykernel_131/2381799523.py:11: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if not pd.api.types.is_period_dtype(dfm["PERIODE"]):


L'ancienneté évolue chaque année pour chaque matricule.


#### Ancienneté dans l'organisme

In [72]:
panel_solde_df, rep = build_anciennete_organisme_depuis_ps_min_collecte(
    panel_solde_df_mois,
    matricule_col="MATRICULE",
    org_col="CODE_ORGANISME",
    collect_col="DATE_COLLECTE",
    ps_col="PRISE DE SERVICE",
    entry_col="DATE_ENTREE_ORG",
    months_col="ANCIENNETE_ORG_MOIS",
    years_col="ANCIENNETE_ORG_AN",
    inclusive=True,
    fallback_when_ps_missing="min_collecte",  # ou "min_ps_non_null"
    return_report=True
)

display(rep["extrait"].head(15))


,MATRICULE,CODE_ORGANISME,PERIODE,DATE_ENTREE_ORG,ANCIENNETE_ORG_MOIS,ANCIENNETE_ORG_AN
0,011242X,15,2015-01-01,1969-01-01,553,46
184890,011242X,15,2015-02-01,1969-01-01,554,46
370226,011242X,15,2015-03-01,1969-01-01,555,46
556520,011242X,15,2015-04-01,1969-01-01,556,46
744494,011242X,15,2015-05-01,1969-01-01,557,46
934474,011242X,15,2015-06-01,1969-01-01,558,46
1126818,011242X,15,2015-07-01,1969-01-01,559,46
1319932,011242X,15,2015-08-01,1969-01-01,560,46
1514281,011242X,15,2015-09-01,1969-01-01,561,46
1709249,011242X,15,2015-10-01,1969-01-01,562,46


#### Vérification de la croissance de l'anciennété 

In [73]:
import pandas as pd

df = panel_solde_df.copy()

matricule_col = "MATRICULE"
org_col       = "CODE_ORGANISME"
collect_col   = "DATE_COLLECTE"
months_col    = "ANCIENNETE_ORG_MOIS"   # déjà créé par ta fonction

# --- Année de référence (à partir de PERIODE si dispo, sinon DATE_COLLECTE) ---
if "PERIODE" in df.columns:
    if not pd.api.types.is_period_dtype(df["PERIODE"]):
        df["PERIODE"] = pd.PeriodIndex(df["PERIODE"].astype(str), freq="M")
    df["ANNEE"] = df["PERIODE"].dt.year
else:
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")
    df["ANNEE"] = df[collect_col].dt.year

# --- Ancienneté de fin d'année par (matricule, organisme, année) ---
year_end = (
    df.groupby([matricule_col, org_col, "ANNEE"], as_index=False)[months_col]
      .max()  # fin d'année = max mensuel dans l'année
      .sort_values([matricule_col, org_col, "ANNEE"])
)

# --- Deltas année→année dans chaque (matricule, organisme) ---
year_end["DELTA"] = (
    year_end.groupby([matricule_col, org_col])[months_col]
            .diff()
)

# --- Règles globales (un seul message) ---
any_negative = (year_end["DELTA"] < 0).any()

evolue_tous = (
    year_end.groupby([matricule_col, org_col])["DELTA"]
            .apply(lambda s: s.dropna().gt(0).all())
            .all()
)

constant_tous = year_end["DELTA"].fillna(0).eq(0).all()

if any_negative:
    print("L'ancienneté decroit.")
elif evolue_tous:
    print("L'ancienneté évolue chaque année pour chaque matricule.")
elif constant_tous:
    print("L'ancienneté est constant chaque année pour chaque matricule.")
else:
    # Cas mixte (ni toute en hausse, ni toute constante, mais sans baisse explicite)
    print("L'ancienneté decroit.")


/tmp/ipykernel_131/3646309407.py:12: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if not pd.api.types.is_period_dtype(df["PERIODE"]):


L'ancienneté decroit.
